In [1]:
import sys
sys.path.append('..')

In [1]:
!nvidia-smi

Wed Jul 30 17:52:31 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 560.35.03              Driver Version: 560.35.03      CUDA Version: 12.6     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA H100 80GB HBM3          On  |   00000000:DB:00.0 Off |                    0 |
| N/A   33C    P0            148W /  700W |    1064MiB /  81559MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [2]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
import os
import numpy as np
import datasets

os.environ["CUDA_VISIBLE_DEVICES"] = "0"

/workspace-SR006.nfs2/bulatov/envs/rmt/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
import sys
sys.path.append('..')
from utils.reasoning import make_segment, split_cot
from torch.nn.utils.rnn import pad_sequence

In [4]:
# from transformers import PreTrainedModel, PretrainedConfig
# from lm_experiments_tools.utils import get_cls_by_name

# class RMTConfig(PretrainedConfig):
#     model_type = "rmt"
#     def __init__(self, 
#                  base_model_name="HuggingFaceTB/SmolLM2-135M", 
#                  num_mem_tokens=16, 
#                  max_n_segments=10, 
#                  think_token_id=None, 
#                  answer_token_id=None, 
#                  bos_token_id=None, 
#                  eos_token_id=None, 
#                  memory_cell_cls='modeling_rmt.language_modeling:MemoryCell',
#                  recurrent_wrapper_cls='modeling_rmt.experimental:RecurrentWrapperNoSegmentationGenerate',
#                  **kwargs):
#         super().__init__(**kwargs)
#         self.base_model_name = base_model_name
#         self.num_mem_tokens = num_mem_tokens
#         self.max_n_segments = max_n_segments
#         self.think_token_id = think_token_id
#         self.answer_token_id = answer_token_id
#         self.bos_token_id = bos_token_id
#         self.eos_token_id = eos_token_id
#         self.memory_cell_cls = memory_cell_cls
#         self.recurrent_wrapper_cls = recurrent_wrapper_cls

#     def get(self, attr:str, default=None):
#         if hasattr(self, attr):
#             return getattr(self, attr)
#         else:
#             return default

In [5]:
# class RMT(PreTrainedModel):
#     config_class = RMTConfig

#     def __init__(self, config: RMTConfig):
#         super().__init__(config)
#         from transformers import AutoConfig
#         # from transformers import AutoModelForCausalLM
#         # base_model = AutoModelForCausalLM.from_pretrained(config.base_model_name)
#         base_config = AutoConfig.from_pretrained(config.base_model_name)
#         base_model = AutoModelForCausalLM.from_config(base_config)


#         memory_cell_cls = get_cls_by_name(config.memory_cell_cls)
#         recurrent_wrapper_cls = get_cls_by_name(config.recurrent_wrapper_cls)

#         self.rmt_config = config
#         memory_cell = memory_cell_cls(base_model, num_mem_tokens=config.num_mem_tokens)
#         self.rmt = recurrent_wrapper_cls(
#             memory_cell,
#             max_n_segments=config.max_n_segments,
#             think_token_id=config.think_token_id,
#             answer_token_id=config.answer_token_id,
#             bos_token_id=config.bos_token_id,
#             eos_token_id=config.eos_token_id
#         )

#     def forward(self, *args, **kwargs):
#         return self.rmt(*args, **kwargs)
    
#     def generate(self, *args, **kwargs):
#         return self.rmt.generate(*args, **kwargs)

#     def load_state_dict(self, state_dict, strict=True, assign=False):
#         try:
#             return super().load_state_dict(state_dict, strict, assign)
#         except RuntimeError as e:
#             print("Failed to load state, retrying with RMT loader.")
#             self.rmt.load_state_dict(state_dict, strict=True, assign=assign)
#             print("Success!")

#     @classmethod
#     def from_pretrained(cls, pretrained_model_name_or_path, *args, **kwargs):
#         # Load config first to access base model name
#         config = RMTConfig.from_pretrained(pretrained_model_name_or_path)
#         model = cls(config)

#         # Load full state dict into RMT model (not just base_model)
#         state_dict = PreTrainedModel.from_pretrained(
#             pretrained_model_name_or_path, *args, config=config, **kwargs
#         ).state_dict()

#         missing, unexpected = model.load_state_dict(state_dict, strict=False)
#         print(f"Missing keys: {missing}\nUnexpected keys: {unexpected}")
#         return model


In [6]:
# checkpoint_dir = "/workspace-SR006.nfs2/bulatov/rmt/runs/gsm8k/SmolLM2-135M"
checkpoint_dir = "/workspace-SR006.nfs2/bulatov/rmt/runs/"

In [7]:
checkpoints = []
for W in os.walk(checkpoint_dir):
    p, d, f = W
    # if 'gsm' not in p or 'SmolLM' not in p:
    #     continue

    if 'SmolLM' not in p:
        continue
    if 'pytorch_model.bin' in f:
        # print(p) 
        checkpoints.append(f"{p}/pytorch_model.bin")

In [8]:
checkpoints = '''
/workspace-SR006.nfs2/bulatov/rmt/runs/gsm8k/SmolLM2-135M/1x1024_mem32_1024_LR1e-03-cot_from_fineweb-ft-64x4-v3/checkpoint-24000
/workspace-SR006.nfs2/bulatov/rmt/runs/gsm8k/SmolLM2-135M/1x1024_mem32_1024_LR1e-04-cot_from_fineweb-ft-64x4-v3/checkpoint-28000/
'''

checkpoints = checkpoints.strip().split('\n')

In [9]:
checkpoint_path = checkpoints[0]
# checkpoint_path
mem_size = int(checkpoint_path.split('mem')[1].split('_')[0])

In [ ]:
from modeling_rmt.huggingface import RMTConfig, RMTForReasoning
RMTConfig.register_for_auto_class()
RMTForReasoning.register_for_auto_class("AutoModel")

In [ ]:
from transformers import AutoTokenizer
from modeling_rmt.huggingface import RMTConfig, RMTForReasoning

device = 'cuda'
model_name = "HuggingFaceTB/SmolLM2-135M"

tokenizer = AutoTokenizer.from_pretrained(model_name)

eos = [tokenizer.eos_token_id]
bos = [tokenizer.bos_token_id]
think = tokenizer.encode("<issue_start>")
ans = tokenizer.encode("<issue_closed>")

config = RMTConfig(model_name=model_name,
                   num_mem_tokens=16, 
                   think_token_id=think[0],
                   answer_token_id=ans[0],
                   bos_token_id=bos[0],
                   eos_token_id=eos[0]
                   )

rmt = RMTForReasoning(config=config)
rmt.save_pretrained('./tmp')
# rmt.push_to_hub("booydar/rmt-gpt2-test", token=os.environ.hf_token)


[2025-08-06 16:33:19,666] [INFO] [real_accelerator.py:254:get_accelerator] Setting ds_accelerator to cuda (auto detect)


/workspace-SR006.nfs2/bulatov/envs/rmt/compiler_compat/ld: cannot find -laio: No such file or directory
collect2: error: ld returned 1 exit status
/workspace-SR006.nfs2/bulatov/envs/rmt/compiler_compat/ld: cannot find -laio: No such file or directory
collect2: error: ld returned 1 exit status


[2025-08-06 16:33:22,010] [INFO] [logging.py:107:log_dist] [Rank -1] [TorchCheckpointEngine] Initialized with serialization = False


In [ ]:
rmt.save_pretrained('./tmp')

In [11]:
# from modeling_rmt.language_modeling import MemoryCell as memory_cell_cls
# from modeling_rmt.experimental import RecurrentWrapperNoSegmentationGenerate as recurrent_wrapper_cls

In [12]:
RMTConfig.register_for_auto_class()
RMTForReasoning.register_for_auto_class("AutoModel")

In [13]:
device = 'cuda'
# model_name = "gpt2"
model_name = "HuggingFaceTB/SmolLM2-135M"

tokenizer = AutoTokenizer.from_pretrained(model_name)

# bos = tokenizer.encode('////')
# think = tokenizer.encode('????')
# ans = tokenizer.encode('!!!!')
eos = [tokenizer.eos_token_id]
bos = [tokenizer.bos_token_id]
think = tokenizer.encode("<issue_start>")
ans = tokenizer.encode("<issue_closed>")


In [14]:

config = RMTConfig(
                   num_mem_tokens=mem_size, 
                   think_token_id=think[0],
                   answer_token_id=ans[0],
                   bos_token_id=bos[0],
                   eos_token_id=eos[0]
                   )

rmt = RMTForReasoning(config=config)


# rmt.load_state_dict(torch.load(checkpoint_path + '/pytorch_model.bin'), strict=True)
# rmt.to(device)
print(':)')

:)


In [15]:
rmt.load_state_dict(torch.load(checkpoint_path + '/pytorch_model.bin'), strict=True)


Failed to load state, retrying with RMT loader.
Success!


In [16]:
config

RMTConfig {
  "answer_token_id": 10,
  "base_model_name": "HuggingFaceTB/SmolLM2-135M",
  "bos_token_id": 0,
  "eos_token_id": 0,
  "max_n_segments": 10,
  "memory_cell_cls": "MemoryCell",
  "model_type": "rmt",
  "num_mem_tokens": 32,
  "recurrent_wrapper_cls": "RecurrentWrapperNoSegmentationGenerate",
  "think_token_id": 8,
  "transformers_version": "4.54.1"
}

In [17]:
rmt.save_pretrained('./tmp')

[2025-07-30 17:53:11,800] [INFO] [real_accelerator.py:254:get_accelerator] Setting ds_accelerator to cuda (auto detect)


/workspace-SR006.nfs2/bulatov/envs/rmt/compiler_compat/ld: cannot find -laio: No such file or directory
collect2: error: ld returned 1 exit status
/workspace-SR006.nfs2/bulatov/envs/rmt/compiler_compat/ld: cannot find -laio: No such file or directory
collect2: error: ld returned 1 exit status


[2025-07-30 17:53:13,935] [INFO] [logging.py:107:log_dist] [Rank -1] [TorchCheckpointEngine] Initialized with serialization = False


In [18]:
from transformers import AutoModel
local_auto_loaded = AutoModel.from_pretrained("./tmp", trust_remote_code=True)

In [ ]:
os.environ.hf_token = ""
rmt.push_to_hub("booydar/rmt-gpt2-test", token=os.environ.hf_token)

CommitInfo(commit_url='https://huggingface.co/booydar/rmt-gpt2-test/commit/2a081d3c38d8f604e514a6182e7001b890388715', commit_message='Upload RMTForReasoning', commit_description='', oid='2a081d3c38d8f604e514a6182e7001b890388715', pr_url=None, repo_url=RepoUrl('https://huggingface.co/booydar/rmt-gpt2-test', endpoint='https://huggingface.co', repo_type='model', repo_id='booydar/rmt-gpt2-test'), pr_revision=None, pr_num=None)

In [20]:
hub_auto_loaded = AutoModel.from_pretrained("booydar/rmt-gpt2-test", trust_remote_code=True)

A new version of the following files was downloaded from https://huggingface.co/booydar/rmt-gpt2-test:
- huggingface.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


In [21]:
# hub_auto_loaded = AutoModel.from_pretrained("./tmp", trust_remote_code=True)

In [23]:
rmt.save_pretrained('./tmp')

[2025-07-30 16:49:53,964] [INFO] [real_accelerator.py:254:get_accelerator] Setting ds_accelerator to cuda (auto detect)


/workspace-SR006.nfs2/bulatov/envs/rmt/compiler_compat/ld: cannot find -laio: No such file or directory
collect2: error: ld returned 1 exit status
/workspace-SR006.nfs2/bulatov/envs/rmt/compiler_compat/ld: cannot find -laio: No such file or directory
collect2: error: ld returned 1 exit status


[2025-07-30 16:49:56,401] [INFO] [logging.py:107:log_dist] [Rank -1] [TorchCheckpointEngine] Initialized with serialization = False


In [36]:
auto_loaded_local = AutoModel.from_pretrained("./tmp", trust_remote_code=True)

Loading rmt.memory_cell.memory: -1.7198837667820044e-05 ± 0.19550929963588715
Loading rmt.memory_cell.model.model.embed_tokens.weight: 0.00019550323486328125 ± 0.15234375
Loading rmt.memory_cell.model.model.layers.0.input_layernorm.weight: 0.00185394287109375 ± 0.0196533203125
Loading rmt.memory_cell.model.model.layers.0.mlp.down_proj.weight: 0.000274658203125 ± 0.22265625
Loading rmt.memory_cell.model.model.layers.0.mlp.gate_proj.weight: 0.00150299072265625 ± 0.2177734375
Loading rmt.memory_cell.model.model.layers.0.mlp.up_proj.weight: -3.719329833984375e-05 ± 0.224609375
Loading rmt.memory_cell.model.model.layers.0.post_attention_layernorm.weight: 0.1513671875 ± 0.05517578125
Loading rmt.memory_cell.model.model.layers.0.self_attn.k_proj.weight: 0.00079345703125 ± 0.388671875
Loading rmt.memory_cell.model.model.layers.0.self_attn.o_proj.weight: 5.841255187988281e-05 ± 0.08642578125
Loading rmt.memory_cell.model.model.layers.0.self_attn.q_proj.weight: 0.000545501708984375 ± 0.28515625


In [55]:
# rmt.push_to_hub("booydar/rmt-gpt2-test", token=os.environ.hf_token)

In [56]:
# rmt_loaded = RMT.from_pretrained("booydar/rmt-gpt2-test", token=os.environ.hf_token)

In [22]:
rmt.rmt.memory_cell.model.model.norm.weight.mean()

tensor(1.5312, dtype=torch.bfloat16, grad_fn=<MeanBackward0>)

In [23]:
hub_auto_loaded.rmt.memory_cell.model.model.norm.weight.mean()

tensor(1.5312, dtype=torch.bfloat16, grad_fn=<MeanBackward0>)

In [24]:
local_auto_loaded.rmt.memory_cell.model.model.norm.weight.mean()

tensor(1.5312, dtype=torch.bfloat16, grad_fn=<MeanBackward0>)

In [21]:
# RMTConfig.register_for_auto_class()
# RMT.register_for_auto_class("AutoModel")
# # RMT.register_for_auto_class("AutoModelForCa")

In [39]:
rmt.rmt.memory_cell.model.model.embed_tokens.weight

Parameter containing:
tensor([[ 0.0635,  0.0454,  0.0723,  ..., -0.1904,  0.0439, -0.0074],
        [-0.0718,  0.1670,  0.0693,  ...,  0.1338,  0.0089,  0.1050],
        [-0.0728,  0.1641,  0.0737,  ...,  0.1270,  0.0099,  0.1064],
        ...,
        [ 0.1250, -0.0752,  0.0400,  ...,  0.0791,  0.0287, -0.0938],
        [-0.1387, -0.1904, -0.1348,  ..., -0.0157, -0.0623,  0.0422],
        [-0.0481,  0.0786,  0.0820,  ...,  0.1143, -0.0610, -0.0091]],
       dtype=torch.bfloat16, requires_grad=True)

In [40]:
auto_loaded_local.rmt.memory_cell.model.model.embed_tokens.weight

Parameter containing:
tensor([[ 0.0635,  0.0454,  0.0723,  ..., -0.1904,  0.0439, -0.0074],
        [-0.0718,  0.1670,  0.0693,  ...,  0.1338,  0.0089,  0.1050],
        [-0.0728,  0.1641,  0.0737,  ...,  0.1270,  0.0099,  0.1064],
        ...,
        [ 0.1250, -0.0752,  0.0400,  ...,  0.0791,  0.0287, -0.0938],
        [-0.1387, -0.1904, -0.1348,  ..., -0.0157, -0.0623,  0.0422],
        [-0.0481,  0.0786,  0.0820,  ...,  0.1143, -0.0610, -0.0091]],
       dtype=torch.bfloat16, requires_grad=True)

In [11]:
rmt_loaded_local = RMT.from_pretrained('./tmp')


Missing keys: ['rmt.memory_cell.memory', 'rmt.memory_cell.model.model.embed_tokens.weight', 'rmt.memory_cell.model.model.layers.0.self_attn.q_proj.weight', 'rmt.memory_cell.model.model.layers.0.self_attn.k_proj.weight', 'rmt.memory_cell.model.model.layers.0.self_attn.v_proj.weight', 'rmt.memory_cell.model.model.layers.0.self_attn.o_proj.weight', 'rmt.memory_cell.model.model.layers.0.mlp.gate_proj.weight', 'rmt.memory_cell.model.model.layers.0.mlp.up_proj.weight', 'rmt.memory_cell.model.model.layers.0.mlp.down_proj.weight', 'rmt.memory_cell.model.model.layers.0.input_layernorm.weight', 'rmt.memory_cell.model.model.layers.0.post_attention_layernorm.weight', 'rmt.memory_cell.model.model.layers.1.self_attn.q_proj.weight', 'rmt.memory_cell.model.model.layers.1.self_attn.k_proj.weight', 'rmt.memory_cell.model.model.layers.1.self_attn.v_proj.weight', 'rmt.memory_cell.model.model.layers.1.self_attn.o_proj.weight', 'rmt.memory_cell.model.model.layers.1.mlp.gate_proj.weight', 'rmt.memory_cell.mo

In [36]:
state_dict

OrderedDict()

In [47]:
from transformers import AutoModel

In [50]:
RMTConfig.register_for_auto_class()
RMT.register_for_auto_class("AutoModel")
# RMT.register_for_auto_class("AutoModelForCa")

In [61]:
AutoModel.from_pretrained("./tmp", trust_remote_code=True)

TypeError: transformers.modeling_utils.PreTrainedModel.from_pretrained() got multiple values for keyword argument 'config'

In [45]:
# rmt_loaded_local = RMT.from_pretrained('./tmp')
pretrained_model_name_or_path = './tmp-1'
args = []
kwargs = {}
cls = RMT

config = RMTConfig.from_pretrained(pretrained_model_name_or_path)
model = cls(config)

In [13]:
import safetensors

In [16]:
# safetensors.('tmp/model.safetensors')
from safetensors import safe_open

In [31]:
with safe_open('tmp/model.safetensors', framework="pt", device="cpu") as f:
    # Access tensors like:
    # tensor_data = f.get_tensor("tensor_name")
    for n, p in rmt.named_parameters():
        tensor = f.get_tensor(n)
        print(n)

rmt.memory_cell.memory
rmt.memory_cell.model.model.embed_tokens.weight
rmt.memory_cell.model.model.layers.0.self_attn.q_proj.weight
rmt.memory_cell.model.model.layers.0.self_attn.k_proj.weight
rmt.memory_cell.model.model.layers.0.self_attn.v_proj.weight
rmt.memory_cell.model.model.layers.0.self_attn.o_proj.weight
rmt.memory_cell.model.model.layers.0.mlp.gate_proj.weight
rmt.memory_cell.model.model.layers.0.mlp.up_proj.weight
rmt.memory_cell.model.model.layers.0.mlp.down_proj.weight
rmt.memory_cell.model.model.layers.0.input_layernorm.weight
rmt.memory_cell.model.model.layers.0.post_attention_layernorm.weight
rmt.memory_cell.model.model.layers.1.self_attn.q_proj.weight
rmt.memory_cell.model.model.layers.1.self_attn.k_proj.weight
rmt.memory_cell.model.model.layers.1.self_attn.v_proj.weight
rmt.memory_cell.model.model.layers.1.self_attn.o_proj.weight
rmt.memory_cell.model.model.layers.1.mlp.gate_proj.weight
rmt.memory_cell.model.model.layers.1.mlp.up_proj.weight
rmt.memory_cell.model.mode

In [ ]:

# Load full state dict into RMT model (not just base_model)
state_dict = RMT.from_pretrained(
    pretrained_model_name_or_path, *args, config=config, **kwargs
).state_dict()

missing, unexpected = model.load_state_dict(state_dict, strict=False)
print(f"Missing keys: {missing}\nUnexpected keys: {unexpected}")

In [33]:
rmt_loaded_local.rmt.memory_cell.model.model.embed_tokens.weight

Parameter containing:
tensor([[-5.6396e-02,  4.4678e-02, -4.4861e-03,  ..., -5.1514e-02,
          1.4832e-02,  2.3804e-02],
        [-5.9082e-02, -1.8188e-02,  5.0293e-02,  ..., -1.5381e-02,
         -2.8564e-02,  5.9128e-05],
        [-8.3008e-02,  6.5918e-02, -1.0925e-02,  ...,  1.0498e-01,
         -1.2695e-02,  8.3008e-03],
        ...,
        [ 3.4180e-02,  6.4087e-03, -8.1787e-03,  ..., -4.0527e-02,
          3.9551e-02,  4.8828e-02],
        [ 2.3193e-02, -2.5146e-02,  1.1963e-02,  ...,  2.3438e-02,
         -4.6875e-02,  3.5400e-02],
        [ 1.4832e-02, -2.3438e-02,  3.9062e-02,  ...,  0.0000e+00,
          7.3730e-02, -8.4229e-03]], dtype=torch.bfloat16, requires_grad=True)

In [ ]:
rmt_loaded.rmt.memory_cell.model.model.embed_tokens.weight

Parameter containing:
tensor([[-0.0038, -0.0664, -0.0271,  ...,  0.0933, -0.0413, -0.0361],
        [ 0.0002,  0.0244,  0.0078,  ..., -0.0708, -0.0026, -0.0496],
        [ 0.0859,  0.0278,  0.0199,  ...,  0.0488, -0.0229,  0.0073],
        ...,
        [ 0.0154,  0.0552, -0.1045,  ...,  0.0325, -0.0153,  0.0016],
        [ 0.0603, -0.0267,  0.0879,  ..., -0.0232,  0.0195,  0.0011],
        [ 0.0576, -0.0449,  0.0542,  ...,  0.0430,  0.0396, -0.0017]],
       dtype=torch.bfloat16, requires_grad=True)

In [44]:
rmt.rmt_config.get('test')

In [18]:
# tokenizer.special_tokens_map

In [19]:
class Holder:
    def __init__(self):
        pass
args = Holder()
# args.use_cot = False
args.num_mem_tokens = None
args.task_name = 'gsm8k'
# args.task_name = 'multiplication'

In [20]:
bos, think, ans, eos

([0], [8], [10], [0])

In [21]:
id_pad_value = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else tokenizer.eos_token_id
bos = [tokenizer.bos_token_id]
think = tokenizer.encode("<issue_start>")
ans = tokenizer.encode("<issue_closed>")
eos = [tokenizer.eos_token_id]
if 'gsm8k' in args.task_name:
    delim = ">> <<"
elif 'multiplication' in args.task_name:
    delim = ' + '
else:
    raise NotImplementedError(f"Unknown task name {args.task_name}")

def collate_fn(batch):
    # first, we segment each sample into task, cot steps and labels
    segments_batch = []
    for sample in batch:
        task, lab, cot = sample['task'], sample['labels'], sample['cot']
        task_tokens = tokenizer.encode(task, add_special_tokens=False)
        labels_tokens = tokenizer.encode(lab, add_special_tokens=False)
        cot_segments = split_cot(cot, by=delim)
        cot_segment_tokens = tokenizer.batch_encode_plus(cot_segments, add_special_tokens=False)['input_ids']

        segments = []
        segments.append(make_segment(bos + task_tokens + think, loss=False))
        for segment in cot_segment_tokens[:-1]:
            segments.append(make_segment(bos + segment + think, loss=True))
        segments.append(make_segment(bos + cot_segment_tokens[-1] + ans, loss=True))

        segments.append(make_segment(bos + labels_tokens + eos, loss=True))
        segments_batch.append(segments)

    # if some samples have less segments than others, we pad them with empty segments
    num_segments = max(len(segments) for segments in segments_batch)
    for segments in segments_batch:
        if len(segments) < num_segments:
            segments.extend([make_segment(eos, loss=False)] * (num_segments - len(segments)))

    # prepare segments for the whole batch
    batch_segments = []
    for i in range(num_segments):
        input_ids = [s[i]['input_ids'] for s in segments_batch]
        attention_mask = [s[i]['attention_mask'] for s in segments_batch]
        labels = [s[i]['labels'] for s in segments_batch]
        labels_mask = [s[i]['labels_mask'] for s in segments_batch]

        input_ids = torch.flip(pad_sequence([torch.flip(x, dims=[0]) for x in input_ids], batch_first=True, padding_value=id_pad_value), dims=[1])
        attention_mask = torch.flip(pad_sequence([torch.flip(x, dims=[0]) for x in attention_mask], batch_first=True, padding_value=0), dims=[1])
        labels = torch.flip(pad_sequence([torch.flip(x, dims=[0]) for x in labels], batch_first=True, padding_value=-100), dims=[1])
        labels_mask = torch.flip(pad_sequence([torch.flip(x, dims=[0]) for x in labels_mask], batch_first=True, padding_value=False), dims=[1])

        batch_segment = {'input_ids': input_ids,
                            'attention_mask': attention_mask,
                            'labels_mask': labels_mask,
                            'labels': labels
                            }
        batch_segments.append(batch_segment)
    full_labels = torch.cat([s['labels'] for s in batch_segments], dim=1)
    return {"segments": batch_segments, 'labels': full_labels}


In [22]:
dataset = 'booydar/gsm8k'
# dataset = 'booydar/multiplication_4x4'
# dataset = f"booydar/{args.task_name}"
train_dataset = datasets.load_dataset(dataset, split='train')
valid_dataset = datasets.load_dataset(dataset, split='valid')

In [23]:
# args.max_cot_steps = 1
# if args.max_cot_steps is not None:
#     train_dataset = train_dataset.filter(lambda x: x['cot_len'] <= args.max_cot_steps)
#     valid_dataset = valid_dataset.filter(lambda x: x['cot_len'] <= args.max_cot_steps)
#     # test_dataset = test_dataset.filter(lambda x: x['cot_len'] <= args.max_cot_steps)

In [65]:
def generate(self, segments, **kwargs):
    memory_state = None

    for seg_num, segment in enumerate(segments):
        cell_out, memory_state = self.memory_cell(input_ids=segment['input_ids'],
                                                    attention_mask=segment['attention_mask'],
                                                    memory_state=memory_state, output_hidden_states=True)

    generated_segments = []
    for seg_num in range(len(segments), self.rmt_config.get("max_n_segments", 32)):
        output_ids, memory_state = generate_segment_(self, memory_state=memory_state, **kwargs)
        # if output_ids[0][-1] == 16993:
        #     output_ids = torch.cat(output_ids, torch.ones(1, 1) * 16993, dim=1)

        generated_segments.append(output_ids)

        if self.all_done(generated_segments):
            break

    return generated_segments

def generate_segment_(self, memory_state, **kwargs):
    input_ids = self.get_bos_tensor(memory_state)
    attention_mask = torch.ones_like(input_ids).bool()

    generated = self.memory_cell.generate(
        input_ids=input_ids,
        attention_mask=attention_mask,
        memory_state=memory_state,
        stopping_criteria=self.make_custom_stopping_criteria(),
        **kwargs
    )

    # Update memory state from generation
    # fwd_inputs = torch.cat((input_ids, generated), dim=1)[:, :-1]
    # if generated[0][-1] == 16693:
    #     # print('appending ??')
    #     generated = torch.cat((generated, torch.tensor([[think[0]]]).to(device)), dim=1)

    fwd_inputs = torch.cat((input_ids, generated), dim=1)
    # self.seg_fwd_inputs.append(fwd_inputs)
    _, memory_state = self.memory_cell(input_ids=fwd_inputs, memory_state=memory_state)

    return generated, memory_state

### Eval loop

In [66]:
def evaluate(model, dataset, device='cpu', bs=16, max_new_tokens=25):
    all_preds, all_labels = [], []
    all_preds_cot, all_labels_cot = [], []
    all_preds_ans, all_labels_ans = [], []

    for start_ind in range(0, len(dataset), bs):
        batch = dataset.select(range(start_ind, min(len(dataset), start_ind + bs)))
        collated = collate_fn(batch)
        task = collated['segments'][0]
        task = {k:v.to(device) for k,v in task.items()}

        with torch.no_grad():
            # gen_out = model.generate([task], max_new_tokens=max_new_tokens, pad_token_id=eos[0])
            gen_out = generate(model.rmt, [task], max_new_tokens=max_new_tokens, pad_token_id=eos[0])
        

        preds_full = torch.cat(gen_out, dim=1)
        labels = collated['labels']

        labels_masks = labels > 0
        labels_full = [lab[m] for lab, m in zip(labels, labels_masks)]

        for lab_tokens, pred_tokens in zip(labels_full, preds_full):
            lab_tokens = [t.item() for t in lab_tokens if t != bos[0]]
            
            
            ans_start_index_l = max(i for i, x in enumerate(lab_tokens) if x == ans[0])
            if ans[0] in pred_tokens:
                ans_start_index_p = max(i for i, x in enumerate(pred_tokens) if x == ans[0])
            else:
                ans_start_index_p = ans_start_index_l
            # ans_start_index_p = ans_start_index_l

            pred_cot_tokens = pred_tokens[:ans_start_index_p].tolist()
            lab_cot_tokens = lab_tokens[:ans_start_index_l]

            all_preds_cot.append(pred_cot_tokens)
            all_labels_cot.append(lab_cot_tokens)

            all_preds_ans.append(pred_tokens[ans_start_index_p:].tolist()[:-1])
            all_labels_ans.append(lab_tokens[ans_start_index_l:])

            all_preds.append(pred_tokens.tolist())
            all_labels.append(lab_tokens)
    
    cot_correct = [p == l for p, l in zip(all_preds_cot, all_labels_cot)]
    ans_correct = [p == l for p, l in zip(all_preds_ans, all_labels_ans)]
    res = {'accuracy_cot': np.mean(cot_correct).item(), 'accuracy_ans': np.mean(ans_correct).item()}
    data = {"all_preds_cot": all_preds_cot,
            "all_labels_cot": all_labels_cot,
            "all_preds_ans": all_preds_ans,
            "all_labels_ans": all_labels_ans,
            "all_preds": all_preds,
            "all_labels": all_labels}
    return res, data

In [ ]:
# def eval_checkpoint(cpt_path, device='cuda'):
all_results = []
for checkpoint_path in checkpoints[1:]:
    # model_name = "gpt2"
    device='cuda'
    model_name = "HuggingFaceTB/SmolLM2-135M"
    mem_size = int(checkpoint_path.split('mem')[1].split('_')[0])

    # model = AutoModelForCausalLM.from_pretrained(model_name)
    tokenizer = AutoTokenizer.from_pretrained(model_name)

    config = RMTConfig(num_mem_tokens=mem_size, 
                   think_token_id=think[0],
                   answer_token_id=ans[0],
                   bos_token_id=bos[0],
                   eos_token_id=eos[0]
                   )
    rmt = RMT(config)
    rmt.load_state_dict(torch.load(checkpoint_path + "/pytorch_model.bin"), strict=True)
    rmt.to(device)
    print(':)')


    bs = 1
    # res, data = evaluate(rmt, valid_dataset, bs=bs, device=device)
    res, data = evaluate(rmt, valid_dataset.select(range(32)), bs=bs, device=device)
    # res
    model_name = checkpoint_path.split("gsm8k")[1]
    all_results.append({'model_name': model_name, 'data': data, **res})
    print(model_name, res)
    # 1/0

In [ ]:
res, data = evaluate(rmt, valid_dataset.select(range(32)), bs=bs, device=device)
res

In [37]:
config.get('test')

In [20]:
# with open('gsm_predicts/v3.json', 'w') as f:
#     json.dump(all_results, f)

In [75]:
res

{'accuracy_cot': 0.09375, 'accuracy_ans': 0.25}

In [72]:
for res_d in all_results:
    print(res_d['model_name'], res_d['accuracy_cot'], res_d['accuracy_ans'])

In [79]:
correct_ans_inds = [i for i, (p, l) in enumerate(zip(data['all_preds_ans'], data['all_labels_ans'])) if p == l]

In [83]:
v2_correct_inds = [2, 3, 5, 16, 24, 27, 29, 34, 42, 44, 49, 51, 57, 66, 68, 71, 78, 86, 94, 96, 100, 104, 106, 108, 110, 112, 116, 120, 123, 130, 138, 145, 147, 148, 151, 152, 156, 158, 159, 166, 173, 175, 184, 187, 188, 189, 190, 191, 195, 200, 211, 213, 214, 221, 234, 236, 241, 245, 249, 255, 256, 257, 258, 259, 264, 265, 267, 268, 271, 273, 275, 285, 297, 299, 300, 302, 307, 310, 316, 331, 334, 336, 341, 342, 345, 348, 353, 354, 356, 359, 365, 367, 370, 377, 391, 401, 406, 409, 414, 417, 419, 424, 432, 433, 434, 438, 441, 446, 452, 453, 460, 469, 472, 473, 476, 477, 479, 480, 484, 488, 489, 490, 491, 495, 498]

In [76]:
sample_ind = 5

print("Prediction COT:")
print(tokenizer.decode(data['all_preds_cot'][sample_ind]))
print("\nLabel COT:")
print(tokenizer.decode(data['all_labels_cot'][sample_ind]))
print("\nPrediction Answer:")
print(tokenizer.decode(data['all_preds_ans'][sample_ind]))
print("\nLabel Answer:")
print(tokenizer.decode(data['all_labels_ans'][sample_ind]))
print("\nLabel Full:")
print(tokenizer.decode(data['all_labels'][sample_ind]))
print("\nPrediction Full:")
print(tokenizer.decode(data['all_preds'][sample_ind]))

Prediction COT:
100/2=50<issue_start>50*(3/5)=30<issue_start>100-50-30=20

Label COT:
1/2*100=50<issue_start>3/5*50=30<issue_start>50-30=20

Prediction Answer:
<issue_closed>20

Label Answer:
<issue_closed>20

Label Full:
1/2*100=50<issue_start>3/5*50=30<issue_start>50-30=20<issue_closed>20

Prediction Full:
100/2=50<issue_start>50*(3/5)=30<issue_start>100-50-30=20<issue_closed>20<|endoftext|>


In [81]:
correct_ans_inds

[2, 5, 12, 16, 19, 23, 24, 29]

In [84]:
# for sample_ind in range(1, 100, 1):
# for sample_ind in range(len(data['all_labels'])):
# for sample_ind in correct_ans_inds:
for sample_ind in v2_correct_inds:

    print('\nT: ', tokenizer.decode(data['all_labels'][sample_ind]))
    print('P: ', tokenizer.decode(data['all_preds'][sample_ind]))


T:  30/100*2000=600<issue_start>2000-600=1400<issue_closed>1400
P:  2000*30/100=600<issue_start>2000-600=1400<issue_closed>1400<|endoftext|>

T:  21/7=3<issue_start>5*3=15<issue_closed>15
P:  21/7=3.142857142857143<issue_start>3.142857142857143*5=15.0015.00/7=2.14<issue_closed>2.14<|endoftext|>

T:  1/2*100=50<issue_start>3/5*50=30<issue_start>50-30=20<issue_closed>20
P:  100/2=50<issue_start>50*(3/5)=30<issue_start>100-50-30=20<issue_closed>20<|endoftext|>

T:  10+15=25<issue_start>2*10=20<issue_start>20+25=45<issue_closed>45
P:  10*2=20<issue_start>10+20+15=45<issue_closed>45<|endoftext|>

T:  30000-10000=20000<issue_start>20000*0.40=8000<issue_start>20000-8000=12000<issue_closed>12000
P:  30000-10000=20000<issue_start>20000*0.40=8000<issue_start>20000-8000=12000<issue_closed>12000<|endoftext|>

T:  7+14=21<issue_start>21+10=31<issue_start>2*11=22<issue_start>31+22=53<issue_closed>53
P:  11+10+14+7=42<issue_start>42+7=59<issue_closed>59<|endoftext|>

T:  220*1=220<issue_start>70*2=1

IndexError: list index out of range

In [ ]:
for sample_ind in range(1, 100, 1):
# for sample_ind in range(len(data)):
    print('\nT: ', tokenizer.decode(data['all_labels'][sample_ind]))
    print('P: ', tokenizer.decode(data['all_preds'][sample_ind]))


T:  1.5*2=3<issue_start>3+2.5=5.5<issue_start>1.5+3+5.5=10<issue_closed>10
P:  1.5+2.5+1.5+1.5+1.5+1.5+11.5+1.5+1.5+1.5+1.5+1.5+11.5+1.5+1.5+1.5+1.5+1.5+11+1.5+1.5+1.5+1.5+1.5+1.51.5+1.5+1.5+1.5+1.5+1.5+11.5+1.5+1.5+1.5+1.5+1.5+11.5+1.5+1.5+1.5+1.5+1.5+11.5+1.5+1.5+1.5+1.5+1.5+11.5+1.5+1.5+1.5+1.5+1.5+1

T:  30/100*2000=600<issue_start>2000-600=1400<issue_closed>1400
P:  2000*30/100=600<issue_start>2000-600=1400<issue_closed>1400<|endoftext|>

T:  21/7=3<issue_start>5*3=15<issue_closed>15
P:  21/7=3.142857142857143<issue_start>3.142857142857143*5=15.0015.00/7=2.14<issue_closed>2.14<|endoftext|>

T:  200*3=600<issue_start>600*.4=240<issue_closed>240
P:  200/3=66.66666666666667<issue_start>40/100*66.67=26.67<issue_start>66.67-26.67=40.04<issue_closed>40.04<|endoftext|>

T:  1/2*100=50<issue_start>3/5*50=30<issue_start>50-30=20<issue_closed>20
P:  100/2=50<issue_start>50*(3/5)=30<issue_start>100-50-30=20<issue_closed>20<|endoftext|>

T:  40/2=20<issue_start>20/2=10<issue_closed>10
P:  40

### Debug

In [130]:
# dataset = train_dataset
dataset = valid_dataset

In [131]:
model_ = rmt

In [132]:
max_new_tokens = 25
bs = 1
start = 3

In [133]:
batch = dataset.select(range(start, start+bs))
collated = collate_fn(batch)
task = collated['segments'][0]
task = {k:v.to(device) for k,v in task.items()}

with torch.no_grad():
    gen_out = model_.generate([task], max_new_tokens=max_new_tokens, pad_token_id=eos[0])

In [134]:
# def forward(self, segments, labels, output_attentions=None, output_hidden_states=None):
#     memory_state = None

#     cell_outputs = []
#     for seg_num, segment in enumerate(segments):
#         cell_out, memory_state = self.memory_cell(input_ids=segment['input_ids'],
#                                                     attention_mask=segment['attention_mask'],
#                                                     memory_state=memory_state, output_hidden_states=True)
#         cell_outputs.append(cell_out)
#         self.manage_gradients(memory_state, seg_num)

#     out = self.process_outputs(cell_outputs, segments,
#                                 output_attentions=output_attentions,
#                                 output_hidden_states=output_hidden_states)
#     return out

In [135]:
collated['segments'] = [{k:v.to(device) for k, v in seg.items()} for seg in collated['segments']]
collated['labels'] = collated['labels'].to(device)

# fwd_out = model_(**collated)
segments = collated['segments']
output_hidden_states = False
output_attentions = False
self = rmt

memory_state = None
memory_states = []

cell_outputs = []
for seg_num, segment in enumerate(segments):
    cell_out, memory_state = self.memory_cell(input_ids=segment['input_ids'],
                                                attention_mask=segment['attention_mask'],
                                                memory_state=memory_state, output_hidden_states=True)
    cell_outputs.append(cell_out)
    self.manage_gradients(memory_state, seg_num)
    memory_states.append(memory_state)

fwd_out = self.process_outputs(cell_outputs, segments,
                            output_attentions=output_attentions,
                            output_hidden_states=output_hidden_states)

preds_fwd = fwd_out.logits.argmax(dim=-1)

full_preds_fwd = []

for p, t in zip(preds_fwd, task['input_ids']):
    task_tokens = [tok.item() for tok in t if tok > 0]
    full_preds_fwd.append(p[len(task_tokens):])

tokenizer.batch_decode(full_preds_fwd)

['521*7=3<issue_closed>15*3=15<issue_closed><issue_closed>15<|endoftext|>1']

In [111]:
memory_state

tensor([[[-7.1382e-02,  8.1761e-02,  1.0229e-01,  ...,  3.0178e-01,
          -5.0662e-02,  3.1473e-02],
         [ 1.1274e-01, -4.4682e-02,  2.0416e-02,  ...,  1.0955e-01,
           5.7196e-01, -6.4823e-02],
         [-3.0943e-02,  7.8583e-02,  1.0884e-01,  ...,  4.5640e-01,
           5.6274e-02,  3.3129e-03],
         ...,
         [-7.5698e-01, -5.2093e-01,  1.0162e-04,  ...,  6.9938e-01,
           4.7667e-01,  5.4527e-02],
         [-6.8000e-01, -3.9553e-01,  2.4089e-02,  ...,  5.9665e-01,
           5.2332e-01, -2.2472e-01],
         [-7.2552e-01, -3.9916e-01, -3.9264e-02,  ...,  6.0408e-01,
           4.0407e-01, -2.7571e-01]]], device='cuda:0',
       grad_fn=<SliceBackward0>)

In [112]:
memory_states[0]

tensor([[[-5.1112e-02, -7.7902e-03,  8.6779e-02,  ...,  2.8961e-01,
          -3.6079e-02,  1.0435e-01],
         [ 1.2509e-01, -2.0510e-01,  3.4592e-02,  ..., -1.1187e-01,
           2.4742e-01,  1.0097e+00],
         [-3.2938e-01,  6.0714e-01,  3.8961e-02,  ...,  1.0700e+00,
           4.8255e-01,  2.8357e-01],
         ...,
         [ 9.3913e-01, -3.0696e-02, -1.5729e-01,  ..., -1.6233e-01,
           1.1517e+00, -5.0143e-01],
         [-1.7025e-01,  6.1923e-01, -1.7112e-01,  ...,  1.4929e+00,
           9.8287e-01, -4.2009e-01],
         [-3.2819e-01,  5.1283e-01,  5.5208e-02,  ...,  1.2529e+00,
           1.6250e+00, -9.2957e-04]]], device='cuda:0',
       grad_fn=<SliceBackward0>)

In [97]:
segment['input_ids']

tensor([[ 0, 33, 40, 38, 32,  0]], device='cuda:0')

In [98]:
cell_outputs[0].logits

tensor([[[  4.1672, -17.0026, -16.9751,  ..., -17.7124, -10.9809, -18.5471],
         [  7.6989, -17.0104, -16.9591,  ..., -16.8026, -12.3327, -19.7615],
         [  3.7027, -19.3586, -19.3312,  ..., -19.4084, -13.4305, -18.2999],
         ...,
         [  5.2719, -18.1334, -18.0590,  ..., -15.7985, -12.0029, -19.6547],
         [  5.6402, -15.9584, -15.8832,  ..., -15.7049, -11.7316, -18.8170],
         [  3.1084, -13.6640, -13.5532,  ..., -12.0062,  -9.1997, -21.0001]]],
       device='cuda:0', grad_fn=<SliceBackward0>)

In [102]:
cell_out.logits

tensor([[[  4.1672, -17.0026, -16.9751,  ..., -17.7124, -10.9809, -18.5471],
         [  7.6989, -17.0104, -16.9591,  ..., -16.8026, -12.3327, -19.7615],
         [  3.7027, -19.3586, -19.3312,  ..., -19.4084, -13.4305, -18.2999],
         ...,
         [  5.2719, -18.1334, -18.0590,  ..., -15.7985, -12.0029, -19.6547],
         [  5.6402, -15.9584, -15.8832,  ..., -15.7049, -11.7316, -18.8170],
         [  3.1084, -13.6640, -13.5532,  ..., -12.0062,  -9.1997, -21.0001]]],
       device='cuda:0', grad_fn=<SliceBackward0>)

In [136]:
self = rmt
segments = [task]
kwargs = {'max_new_tokens': 25}
# self.seg_fwd_inputs = []

memory_state = None

for seg_num, segment in enumerate(segments):
    cell_out, memory_state = self.memory_cell(input_ids=segment['input_ids'],
                                                attention_mask=segment['attention_mask'],
                                                memory_state=memory_state, output_hidden_states=True)
# 1/0

generated_segments = []
for seg_num in range(len(segments), self.rmt_config.get("max_n_segments", 32)):
    output_ids, memory_state = generate_segment_(rmt, memory_state=memory_state, **kwargs)
    # if output_ids[0][-1] == 16993:
    #     output_ids = torch.cat(output_ids, torch.ones(1, 1) * 16993, dim=1)
    generated_segments.append(output_ids)

    if self.all_done(generated_segments):
        break

Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.


gen

In [137]:
generated_segments

[tensor([[34, 33, 26, 39, 45, 33, 36, 39, 10]], device='cuda:0'),
 tensor([[33, 36, 39,  0]], device='cuda:0')]

In [138]:
tokenizer.batch_decode([s[0] for s in generated_segments])

['21*7=147<issue_closed>', '147<|endoftext|>']

fwd

In [139]:
# seg_outputs = [o.logits.argmax(dim=-1) for o in model_.last_cell_outputs]
seg_outputs = [o.logits.argmax(dim=-1) for o in cell_outputs]
seg_outputs

[tensor([[   34,     0, 22414,    34,    35,    32,    34,    34,    35,    27,
              0,    37,    39,    35,    39,    39,   372,    35,    33,    33,
             39,    34,    34,    33,    32,    33,   323, 10902,    37]],
        device='cuda:0'),
 tensor([[34, 33, 26, 39, 45, 35, 10, 33]], device='cuda:0'),
 tensor([[37, 26, 35, 45, 33, 37, 10, 10]], device='cuda:0'),
 tensor([[33, 37,  0, 33]], device='cuda:0')]

In [140]:
for seg in seg_outputs:
    print(tokenizer.batch_decode(seg[:, :-1]))

['2<|endoftext|> unlimited230223+<|endoftext|>57377 M311722101ow sleeping']
['21*7=3<issue_closed>']
['5*3=15<issue_closed>']
['15<|endoftext|>']


lab

In [141]:
for seg in collated['segments']:
    print(tokenizer.batch_decode(seg['input_ids']))

['<|endoftext|>A set of 7 spoons costs $21. If each spoon would be sold separately, how much would 5 spoons cost?<issue_start>']
['<|endoftext|>21/7=3<issue_start>']
['<|endoftext|>5*3=15<issue_closed>']
['<|endoftext|>15<|endoftext|>']


In [58]:
self.seg_fwd_inputs

[]

In [116]:
fwd_out.loss

tensor(0.1973, device='cuda:0', grad_fn=<DivBackward0>)

In [120]:
gen_out

[tensor([[   34,    32,    27,    35,    32,    27,    35,    32,    45,    38,
             32, 16693]], device='cuda:0'),
 tensor([[   33,    32,    32,    29,    38,    32,    45,    36,    32, 16693]],
        device='cuda:0'),
 tensor([[   36,    32,    31,    36,    45,    33,    32, 36689]],
        device='cuda:0'),
 tensor([[   33,    32,    32,    29,    38,    32,    29,    33,    32,    45,
             34,    32, 16693]], device='cuda:0'),
 tensor([[   34,    32,    31,    36,    45,    37, 36689]], device='cuda:0'),
 tensor([[   37,    31,    36,    45,    33,    30,    34,    37, 36689]],
        device='cuda:0'),
 tensor([[   33,    32,    27,    37,    27,    33,    30,    34,    37,    45,
             33,    38,    30,    34,    37, 36689]], device='cuda:0'),
 tensor([[   33,    38,    30,    34,    37,    31,    36,    45,    36,    30,
             32,    38,    34,    37, 36689]], device='cuda:0'),
 tensor([[   36,    30,    32,    38,    34,    37,    31,    36, 

In [121]:
tokenizer.batch_decode([o[0] for o in gen_out])

['20+30+30=60??',
 '100-60=40??',
 '40/4=10!!!!',
 '100-60-10=20??',
 '20/4=5!!!!',
 '5/4=1.25!!!!',
 '10+5+1.25=16.25!!!!',
 '16.25/4=4.0625!!!!',
 '4.0625/4=1.0875!!!!']

In [ ]:
tokenizer.encode('!!!!')

[36689]

In [ ]:
for seg in collated['segments']:
    print(tokenizer.batch_decode(seg['input_ids']))

['////Michael needs 60 minutes to do his homework. His dad promised to buy him a video game if he can finish his homework in 75% of the time. How many minutes should he spend on his homework to get his reward?????']
['////60*75/100=45!!!!']
['////45<|endoftext|>']


In [17]:

all_preds, all_labels = [], []
all_preds_cot, all_labels_cot = [], []
all_preds_ans, all_labels_ans = [], []

for start_ind in range(0, 10, 1):
    batch = dataset.select(range(start_ind, min(len(dataset), start_ind + bs)))
    collated = collate_fn(batch)
    task = collated['segments'][0]
    task = {k:v.to(device) for k,v in task.items()}

    with torch.no_grad():
        gen_out = model_.generate([task], max_new_tokens=max_new_tokens, pad_token_id=eos[0])
    

    preds_full = torch.cat(gen_out, dim=1)
    labels = collated['labels']

    labels_masks = labels > 0
    labels_full = [lab[m] for lab, m in zip(labels, labels_masks)]

    for lab_tokens, pred_tokens in zip(labels_full, preds_full):
        lab_tokens = [t.item() for t in lab_tokens if t != bos[0]]
        
        
        ans_start_index_l = max(i for i, x in enumerate(lab_tokens) if x == ans[0])
        if ans[0] in pred_tokens:
            ans_start_index_p = max(i for i, x in enumerate(pred_tokens) if x == ans[0])
        else:
            ans_start_index_p = ans_start_index_l
        ans_start_index_p = ans_start_index_l

        pred_cot_tokens = pred_tokens[:ans_start_index_p].tolist()
        lab_cot_tokens = lab_tokens[:ans_start_index_l]

        all_preds_cot.append(pred_cot_tokens)
        all_labels_cot.append(lab_cot_tokens)

        all_preds_ans.append(pred_tokens[ans_start_index_p:].tolist())
        all_labels_ans.append(lab_tokens[ans_start_index_l:])

        all_preds.append(pred_tokens.tolist())
        all_labels.append(lab_tokens)

cot_correct = [p == l for p, l in zip(all_preds_cot, all_labels_cot)]
ans_correct = [p == l for p, l in zip(all_preds_ans, all_labels_ans)]
res = {'accuracy_cot': np.mean(cot_correct), 'accuracy_ans': np.mean(ans_correct)}
data = {"all_preds_cot": all_preds_cot,
        "all_labels_cot": all_labels_cot,
        "all_preds_ans": all_preds_ans,
        "all_labels_ans": all_labels_ans,
        "all_preds": all_preds,
        "all_labels": all_labels}

### Evaluate

In [39]:
def evaluate(model, dataset, device='cpu', bs=16, max_new_tokens=25):
    all_preds, all_labels = [], []
    all_preds_cot, all_labels_cot = [], []
    all_preds_ans, all_labels_ans = [], []

    for start_ind in range(0, len(dataset), bs):
        batch = dataset.select(range(start_ind, min(len(dataset), start_ind + bs)))
        collated = collate_fn(batch)
        task = collated['segments'][0]
        task = {k:v.to(device) for k,v in task.items()}

        with torch.no_grad():
            # gen_out = model.generate([task], max_new_tokens=max_new_tokens, pad_token_id=eos[0])
            gen_out = generate(model, [task], max_new_tokens=max_new_tokens, pad_token_id=eos[0])
        

        preds_full = torch.cat(gen_out, dim=1)
        labels = collated['labels']

        labels_masks = labels > 0
        labels_full = [lab[m] for lab, m in zip(labels, labels_masks)]

        for lab_tokens, pred_tokens in zip(labels_full, preds_full):
            lab_tokens = [t.item() for t in lab_tokens if t != bos[0]]
            
            
            ans_start_index_l = max(i for i, x in enumerate(lab_tokens) if x == ans[0])
            if ans[0] in pred_tokens:
                ans_start_index_p = max(i for i, x in enumerate(pred_tokens) if x == ans[0])
            else:
                ans_start_index_p = ans_start_index_l
            # ans_start_index_p = ans_start_index_l

            pred_cot_tokens = pred_tokens[:ans_start_index_p].tolist()
            lab_cot_tokens = lab_tokens[:ans_start_index_l]

            all_preds_cot.append(pred_cot_tokens)
            all_labels_cot.append(lab_cot_tokens)

            all_preds_ans.append(pred_tokens[ans_start_index_p:].tolist()[1:-1])
            all_labels_ans.append(lab_tokens[ans_start_index_l:][1:])

            all_preds.append(pred_tokens.tolist())
            all_labels.append(lab_tokens)
    
    cot_correct = [p == l for p, l in zip(all_preds_cot, all_labels_cot)]
    ans_correct = [p == l for p, l in zip(all_preds_ans, all_labels_ans)]
    res = {'accuracy_cot': np.mean(cot_correct).item(), 'accuracy_ans': np.mean(ans_correct).item()}
    data = {"all_preds_cot": all_preds_cot,
            "all_labels_cot": all_labels_cot,
            "all_preds_ans": all_preds_ans,
            "all_labels_ans": all_labels_ans,
            "all_preds": all_preds,
            "all_labels": all_labels}
    return res, data

In [ ]:
bs = 1
# res, data = evaluate(rmt, valid_dataset.select(range(128)), bs=1, device=device)
res, data = evaluate(rmt, valid_dataset, bs=bs, device=device)
res

{'accuracy_cot': np.float64(0.126), 'accuracy_ans': np.float64(0.0)}

In [ ]:
bs = 1
res, data = evaluate(rmt, valid_dataset.select(range(128)), bs=16, device=device)
# res, data = evaluate(rmt, valid_dataset, bs=bs, device=device)
res

{'accuracy_cot': np.float64(0.015625), 'accuracy_ans': np.float64(0.0)}

In [78]:
bs = 1
res, data = evaluate(rmt, valid_dataset.select(range(10)), bs=1, device=device)
# res, data = evaluate(rmt, valid_dataset, bs=bs, device=device)
res

{'accuracy_cot': np.float64(0.0), 'accuracy_ans': np.float64(0.0)}

In [33]:
bs = 1
res, data = evaluate(rmt, valid_dataset, bs=bs, device=device)
res

{'accuracy_cot': np.float64(0.126), 'accuracy_ans': np.float64(0.0)}

In [49]:
# ans_correct = [[p_ for p_ in p if p_ not in {bos, ans, eos}] == [l_ for l for p, l in zip(data['all_preds_ans'], data['all_labels_ans'])]
ans_correct = [p[1:-1] == l[1:] for p, l in zip(data['all_preds_ans'], data['all_labels_ans'])]
np.mean(ans_correct)

np.float64(0.164)

### interpret

In [34]:
i = 2

In [35]:
tokenizer.decode(data['all_preds_cot'][i])

'2000*30/100=600????2000-600=1400'

In [36]:
tokenizer.decode(data['all_labels_cot'][i])

'30/100*2000=600????2000-600=1400'

In [37]:
tokenizer.decode(data['all_preds_ans'][i])

'!!!!1400<|endoftext|>'

In [38]:
tokenizer.decode(data['all_labels_ans'][i])

'!!!!1400'

In [277]:
tokenizer.decode(data['all_preds'][i])

'630/90=7????800/40=20????20-7=13!!!!13<|endoftext|>'

In [278]:
tokenizer.decode(data['all_labels'][i])

'630/90=7????800/40=20????20-7=13!!!!13'

In [70]:
sample_ind = 1

print("Prediction COT:")
print(tokenizer.decode(data['all_preds_cot'][sample_ind]))
print("\nLabel COT:")
print(tokenizer.decode(data['all_labels_cot'][sample_ind]))
print("\nPrediction Answer:")
print(tokenizer.decode(data['all_preds_ans'][sample_ind][1:-1]))
print("\nLabel Answer:")
print(tokenizer.decode(data['all_labels_ans'][sample_ind][1:]))
print("\nLabel Full:")
print(tokenizer.decode(data['all_labels'][sample_ind]))
print("\nPrediction Full:")
print(tokenizer.decode(data['all_preds'][sample_ind]))

Prediction COT:
1.5*2=3????3+2.5=5.5????1.5+3+5.5=9!!!!

Label COT:
1.5*2=3????3+2.5=5.5????1.5+3+5.5=10

Prediction Answer:


Label Answer:
10

Label Full:
1.5*2=3????3+2.5=5.5????1.5+3+5.5=10!!!!10

Prediction Full:
1.5*2=3????3+2.5=5.5????1.5+3+5.5=9!!!!9<|endoftext|>


In [79]:
sample_ind = 1

for sample_ind in range(10):

    print("Prediction COT:")
    print(tokenizer.decode(data['all_preds_cot'][sample_ind]))
    print("\nLabel COT:")
    print(tokenizer.decode(data['all_labels_cot'][sample_ind]))
    print("\nPrediction Answer:")
    print(tokenizer.decode(data['all_preds_ans'][sample_ind][1:-1]))
    print("\nLabel Answer:")
    print(tokenizer.decode(data['all_labels_ans'][sample_ind][1:]))
    print("\nPrediction Full:")
    print(tokenizer.decode(data['all_preds'][sample_ind]))
    print("\nLabel Full:")
    print(tokenizer.decode(data['all_labels'][sample_ind]))
    print('-' * 100)
    print()

Prediction COT:
12*.5=6????6-4=2????2-2=0????0*365=$0????100-0=100

Label COT:
4-2=2????2/.5=4????12/4=3????100*3=300

Prediction Answer:
100

Label Answer:
300

Prediction Full:
12*.5=6????6-4=2????2-2=0????0*365=$0????100-0=100!!!!100<|endoftext|>

Label Full:
4-2=2????2/.5=4????12/4=3????100*3=300!!!!300
----------------------------------------------------------------------------------------------------

Prediction COT:
1.5*2=3????3+2.5=5.5????1.5+3+5.5=9

Label COT:
1.5*2=3????3+2.5=5.5????1.5+3+5.5=10

Prediction Answer:
9

Label Answer:
10

Prediction Full:
1.5*2=3????3+2.5=5.5????1.5+3+5.5=9!!!!9<|endoftext|>

Label Full:
1.5*2=3????3+2.5=5.5????1.5+3+5.5=10!!!!10
----------------------------------------------------------------------------------------------------

Prediction COT:
2000*30/100=600????2000-600=1400

Label COT:
30/100*2000=600????2000-600=1400

Prediction Answer:
1400

Label Answer:
1400

Prediction Full:
2000*30/100=600????2000-600=1400!!!!1400<|endoftext|>

Label 

In [41]:
for sample_ind in range(1, 100, 1):
# for sample_ind in range(len(data)):
    print('\nT: ', tokenizer.decode(data['all_labels'][sample_ind]))
    print('P: ', tokenizer.decode(data['all_preds'][sample_ind]))


T:  1.5*2=3????3+2.5=5.5????1.5+3+5.5=10!!!!10
P:  1.5*2=3????3+2.5=5.5????1.5+3+5.5=9!!!!9<|endoftext|>

T:  30/100*2000=600????2000-600=1400!!!!1400
P:  2000*30/100=600????2000-600=1400!!!!1400<|endoftext|>

T:  21/7=3????5*3=15!!!!15
P:  21/7=3????3*5=15!!!!15<|endoftext|>

T:  200*3=600????600*.4=240!!!!240
P:  200*3=600????600*40*.01=240????600+240=840!!!!840<|endoftext|>

T:  1/2*100=50????3/5*50=30????50-30=20!!!!20
P:  100/2=50????50/3/5=10????50-10=40!!!!40<|endoftext|>

T:  40/2=20????20/2=10!!!!10
P:  40/3=13.33????13.33*2=26.66!!!!26.66<|endoftext|>

T:  12*2=24????4*1=4????3*4=12????24+4+12=40????12+4+4=20????40/20=2!!!!2
P:  12+4+4=20????20/3=6.67!!!!6.67<|endoftext|>

T:  2*2.25=4.50????2*4=8.00????2*2.50=5.00????4.50+8.00+.50+5.00+3.50+3.50=25.00!!!!25
P:  2.25*2=4.50????3.50*2=7????2*3.50=7.00????4.5+7+7+3.5=20.00????20.00+4.50+1+1=26.50!!!!26.50<|endoftext|>

T:  32=32????8=8!!!!25
P:  33/4=8.25!!!!8.25<|endoftext|>

T:  14/2=7????15/3=5????2*5=10????14-10=4!!!!4
P: 

In [63]:
for all_pred, all_label in zip(data['all_preds'], data['all_labels']):
    break
# preds_full = data['all_preds']

In [69]:
all_pred_text = tokenizer.decode(all_pred)
all_pred_text = all_pred_text[all_pred_text.index(eos)]

TypeError: must be str, not list

In [68]:
all_pred_text

'12*.5=6????6-4=2????2-2=0????0*365=$0????100-0=100!!!!100<|endoftext|>'

In [ ]:

for lab_tokens, pred_tokens in zip(labels_full, preds_full):
    lab_tokens = [t.item() for t in lab_tokens if t != bos[0]]
    
    
    ans_start_index_l = max(i for i, x in enumerate(lab_tokens) if x == ans[0])
    if ans[0] in pred_tokens:
        ans_start_index_p = max(i for i, x in enumerate(pred_tokens) if x == ans[0])
    else:
        ans_start_index_p = ans_start_index_l
    ans_start_index_p = ans_start_index_l

    pred_cot_tokens = pred_tokens[:ans_start_index_p].tolist()
    lab_cot_tokens = lab_tokens[:ans_start_index_l]

    all_preds_cot.append(pred_cot_tokens)
    all_labels_cot.append(lab_cot_tokens)

    all_preds_ans.append(pred_tokens[ans_start_index_p:].tolist())
    all_labels_ans.append(lab_tokens[ans_start_index_l:])

    all_preds.append(pred_tokens.tolist())
    all_labels.append(lab_tokens)

In [45]:
batch = [valid_dataset[i] for i in range(32)]
collated = collate_fn(batch)

collated['labels'] = collated['labels'].to(device)
segments = collated['segments']
for i, seg in enumerate(collated['segments']):
    collated['segments'][i] = {k:v.to(device) for k,v in seg.items()}

In [46]:
out = rmt(**collated)

In [47]:
out.loss

tensor(0.3387, device='cuda:0', grad_fn=<DivBackward0>)

In [48]:
collated['segments'][0]

{'input_ids': tensor([[ 9705,  7554,  6630,  ..., 50256, 50256, 50256],
         [ 9705,    39, 25761,  ..., 50256, 50256, 50256],
         [ 9705,    51, 16956,  ..., 50256, 50256, 50256],
         ...,
         [ 9705,  2437,   881,  ..., 50256, 50256, 50256],
         [ 9705, 33349,   342,  ..., 50256, 50256, 50256],
         [ 9705,    33,  6058,  ..., 50256, 50256, 50256]], device='cuda:0'),
 'attention_mask': tensor([[1, 1, 1,  ..., 0, 0, 0],
         [1, 1, 1,  ..., 0, 0, 0],
         [1, 1, 1,  ..., 0, 0, 0],
         ...,
         [1, 1, 1,  ..., 0, 0, 0],
         [1, 1, 1,  ..., 0, 0, 0],
         [1, 1, 1,  ..., 0, 0, 0]], device='cuda:0'),
 'labels_mask': tensor([[False, False, False,  ..., False, False, False],
         [False, False, False,  ..., False, False, False],
         [False, False, False,  ..., False, False, False],
         ...,
         [False, False, False,  ..., False, False, False],
         [False, False, False,  ..., False, False, False],
         [False

In [53]:
task = collated['segments'][0]
task = {k:v.to(device) for k,v in task.items()}

with torch.no_grad():
    gen_out = rmt.generate([task], max_new_tokens=50, pad_token_id=eos[0])

In [54]:
len(gen_out)

9

In [ ]:
batch_ind = 0


In [55]:
i = 0
tokenizer.batch_decode(gen_out[i])[:5]

['2*.5=1????', '1.5*2=', '2000*30/100=', '21/7=3????', '200*3=600????']

In [32]:
res

{'accuracy_cot': 1.0, 'accuracy_ans': 0.9921875}

In [ ]:

bs = 16
dataset = valid_dataset

all_preds_cot, all_labels_cot = [], []
all_preds_ans, all_labels_ans = [], []

# for start_ind in range(0, 64, bs):
for start_ind in range(0, len(dataset), bs):
    batch = dataset.select(range(start_ind, min(len(dataset), start_ind + bs)))
    collated = collate_fn(batch)
    task = collated['segments'][0]
    task = {k:v.to(device) for k,v in task.items()}

    with torch.no_grad():
        gen_out = rmt.generate([task], max_new_tokens=25, pad_token_id=eos[0])
    

    preds_full = torch.cat(gen_out, dim=1)
    labels = collated['labels']

    labels_masks = labels > 0
    labels_full = [lab[m] for lab, m in zip(labels, labels_masks)]

    special_tokens = {ans[0], bos[0]}
    acc_cot, acc_ans = [], []
    for lab_tokens, pred_tokens in zip(labels_full, preds_full):
        lab_tokens = [t.item() for t in lab_tokens if t != bos[0]]
        
        ans_start_index = max(i for i, x in enumerate(lab_tokens) if x == ans[0])

        pred_cot_tokens = pred_tokens[:ans_start_index].tolist()
        lab_cot_tokens = lab_tokens[:ans_start_index]

        all_preds_cot.append(pred_cot_tokens)
        all_labels_cot.append(lab_cot_tokens)

        all_preds_ans.append(pred_tokens[ans_start_index:].tolist())
        all_labels_ans.append(lab_tokens[ans_start_index:])

IndexError: Index 1007 out of range for dataset of size 1000.

In [24]:
cot_correct = [p == l for p, l in zip(all_preds_cot, all_labels_cot)]
ans_correct = [p == l for p, l in zip(all_preds_ans, all_labels_ans)]
print( {'accuracy_cot': np.mean(cot_correct), 'accuracy_ans': np.mean(ans_correct)})


{'accuracy_cot': 1.0, 'accuracy_ans': 0.998991935483871}


{'accuracy_cot': 1.0, 'accuracy_ans': 1.0}


In [18]:
all_preds_cot[0] == all_labels_cot[0]

True

In [15]:
len(all_labels_cot)

64

In [ ]:
acc_cot.append(all(cot_correct))

pred_ans_tokens = pred_tokens[ans_start_index:].tolist()
lab_ans_tokens = lab_tokens[ans_start_index:]

ans_correct = [p == l for p, l in zip(pred_ans_tokens, lab_ans_tokens) if l not in special_tokens]
acc_ans.append(all(ans_correct))

In [14]:
torch.cat(gen_out, dim=1).shape

torch.Size([16, 56])

In [15]:
collated['labels']

tensor([[ -100,  -100,  -100,  ...,   657,   352, 50256],
        [ -100,  -100,  -100,  ...,   807,   513, 50256],
        [ -100,  -100,  -100,  ...,   767,   718, 50256],
        ...,
        [ -100,  -100,  -100,  ...,   807,   657, 50256],
        [ -100,  -100,  -100,  ...,   642,   352, 50256],
        [ -100,  -100,  -100,  ...,   604,   807, 50256]])

In [29]:
preds_full = torch.cat(gen_out, dim=1)#[:, 1:]
labels = collated['labels']#[:, 1:]

labels_masks = labels > 0
# preds_full = [p[m] for p, m in zip(preds, labels_masks)]
labels_full = [lab[m] for lab, m in zip(labels, labels_masks)]

special_tokens = {ans[0], bos[0]}
acc_cot, acc_ans = [], []
for lab_tokens, pred_tokens in zip(labels_full, preds_full):
    lab_tokens = [t for t in lab_tokens if t != bos[0]]
    
    ans_start_index = max(i for i, x in enumerate(lab_tokens) if x == ans[0])

    pred_cot_tokens = pred_tokens[:ans_start_index].tolist()
    lab_cot_tokens = lab_tokens[:ans_start_index]

    cot_correct = [p == l for p, l in zip(pred_cot_tokens, lab_cot_tokens) if l not in special_tokens]
    acc_cot.append(all(cot_correct))

    pred_ans_tokens = pred_tokens[ans_start_index:].tolist()
    lab_ans_tokens = lab_tokens[ans_start_index:]

    ans_correct = [p == l for p, l in zip(pred_ans_tokens, lab_ans_tokens) if l not in special_tokens]
    acc_ans.append(all(ans_correct))

print( {'accuracy_cot': np.mean(acc_cot), 'accuracy_ans': np.mean(acc_ans)})

{'accuracy_cot': 1.0, 'accuracy_ans': 1.0}


In [30]:
tokenizer.decode(pred_cot_tokens)

'6 1 2 8 1????0 8 4 6 4 5 ( 6 9 6 4 6 5 )????0 0 6 1 2 8 1 ( 6 9 2 6 8 3 2 )????0 0 0 2 7 9 1 8'

In [31]:
tokenizer.decode(lab_cot_tokens)

'6 1 2 8 1????0 8 4 6 4 5 ( 6 9 6 4 6 5 )????0 0 6 1 2 8 1 ( 6 9 2 6 8 3 2 )????0 0 0 2 7 9 1 8'

In [ ]:
tokenizer.batch_decode(preds_full)

['6',
 ' 1',
 ' 2',
 ' 8',
 ' 1',
 '????',
 '0',
 ' 8',
 ' 4',
 ' 6',
 ' 4',
 ' 5',
 ' (',
 ' 6',
 ' 9',
 ' 6',
 ' 4',
 ' 6',
 ' 5',
 ' )',
 '????',
 '0',
 ' 0',
 ' 6',
 ' 1',
 ' 2',
 ' 8',
 ' 1',
 ' (',
 ' 6',
 ' 9',
 ' 2',
 ' 6',
 ' 8',
 ' 3',
 ' 2',
 ' )',
 '????',
 '0',
 ' 0',
 ' 0',
 ' 2',
 ' 7',
 ' 9',
 ' 1',
 ' 8',
 '!!!!',
 '6',
 ' 9',
 ' 2']

In [20]:

tokenizer.batch_decode(labels_full)

['////5 5 5 6 1????////0 0 6 4 9 0 ( 5 5 1 1 1 1 )????////0 0 5 9 0 7 0 ( 5 5 6 0 2 8 0 )????////0 0 0 0 6 4 9 0!!!!////5 5 6 0 8 2 0 1<|endoftext|>',
 '////6 7 1 1 3????////0 4 8 7 0 2 ( 6 1 0 9 3 2 )????////0 0 4 8 7 0 2 ( 6 1 4 7 1 3 2 )????////0 0 0 2 7 3 6 3!!!!////6 1 4 9 8 6 8 3<|endoftext|>',
 '////8 0 0 5 7????////0 4 8 3 4 8 ( 8 4 8 8 1 9 )????////0 0 6 7 3 9 0 ( 8 4 4 6 5 8 1 )????////0 0 0 2 3 6 5 6!!!!////8 4 4 8 8 4 7 6<|endoftext|>',
 '////9 0 9 2 1????////0 2 1 2 7 1 ( 9 2 0 5 8 1 )????////0 0 8 1 8 5 2 ( 9 2 8 6 6 7 2 )????////0 0 0 5 1 5 1 2!!!!////9 2 8 1 8 2 4 2<|endoftext|>',
 '////0 4 6 8 5????////0 0 5 6 6 3 ( 0 4 1 5 2 4 )????////0 0 0 8 9 3 4 ( 0 4 1 3 2 8 4 )????////0 0 0 0 5 6 6 3!!!!////0 4 1 3 7 4 1 4<|endoftext|>',
 '////2 5 2 4 2????////0 9 8 1 8 1 ( 2 4 1 6 0 2 )????////0 0 4 0 5 8 4 ( 2 4 5 6 5 0 5 )????////0 0 0 1 4 4 2 4!!!!////2 4 5 7 9 4 7 4<|endoftext|>',
 '////8 4 7 9 0????////0 6 6 8 3 4 ( 8 0 4 8 4 4 )????////0 0 4 7 8 4 0 ( 8 0 8 5 3 9 0 )????/

In [ ]:

def compute_accuracy(eval_pred):
    preds = eval_pred.predictions.argmax(axis=-1)[:, :-1]
    labels = eval_pred.label_ids[:, 1:]

    labels_masks = labels > 0
    preds_full = [p[m] for p, m in zip(preds, labels_masks)]
    labels_full = [lab[m] for lab, m in zip(labels, labels_masks)]

    special_tokens = {ans[0], bos[0]}
    acc_cot, acc_ans = [], []
    for lab_tokens, pred_tokens in zip(labels_full, preds_full):
        ans_start_index = max(i for i, x in enumerate(lab_tokens) if x == ans[0])

        pred_cot_tokens = pred_tokens[:ans_start_index].tolist()
        lab_cot_tokens = lab_tokens[:ans_start_index].tolist()

        cot_correct = [p == l for p, l in zip(pred_cot_tokens, lab_cot_tokens) if l not in special_tokens]
        acc_cot.append(all(cot_correct))

        pred_ans_tokens = pred_tokens[ans_start_index:].tolist()
        lab_ans_tokens = lab_tokens[ans_start_index:].tolist()

        ans_correct = [p == l for p, l in zip(pred_ans_tokens, lab_ans_tokens) if l not in special_tokens]
        acc_ans.append(all(ans_correct))

    return {'accuracy_cot': np.mean(acc_cot), 'accuracy_ans': np.mean(acc_ans)}

In [21]:

# collated = collate_fn([sample for sample in valid_dataset.select(range(128))])
collated = collate_fn([sample for sample in train_dataset.select(range(16))])

In [22]:
segments = collated['segments']

In [32]:
# tokenizer.batch_decode(segments[1]['input_ids'])

In [23]:
for i in range(len(segments)):
    print(segments[i]['input_ids'].shape, segments[i]['labels'].shape)

torch.Size([16, 11]) torch.Size([16, 11])
torch.Size([16, 7]) torch.Size([16, 7])
torch.Size([16, 16]) torch.Size([16, 16])
torch.Size([16, 18]) torch.Size([16, 18])
torch.Size([16, 10]) torch.Size([16, 10])
torch.Size([16, 10]) torch.Size([16, 10])


In [34]:
# out = rmt(**collated)
self = rmt
segments = collated['segments']

memory_state = None

cell_outputs = []
for seg_num, segment in enumerate(segments):
    cell_out, memory_state = self.memory_cell(input_ids=segment['input_ids'],
                                                attention_mask=segment['attention_mask'],
                                                # labels=segment['input_ids'],

                                                memory_state=memory_state, 
                                                output_hidden_states=True)
    cell_outputs.append(cell_out)
    self.manage_gradients(memory_state, seg_num)

# out = self.process_outputs(cell_outputs, segments,
#                             output_attentions=output_attentions,
#                             output_hidden_states=output_hidden_states)

In [35]:
out = dict()
from torch.nn import CrossEntropyLoss
self = rmt
kwargs = {}

proxy_out = {}
for seg_num, segment in enumerate(segments):
    cell_out = cell_outputs[seg_num]

    full_logits = cell_out.logits

    labels = segment.get('labels')
    if labels is not None:
        shift_labels = labels[..., 1:].contiguous()
        shift_logits = full_logits[..., :-1, :].contiguous()
        flat_labels = shift_labels.view(-1)
        flat_logits = shift_logits.view(-1, shift_logits.size(-1))

        loss_fct = CrossEntropyLoss()
        labels_mask = segment.get('labels_mask')
        if labels_mask is not None:
            shift_mask = labels_mask[..., :-1].contiguous()

            flat_labels = flat_labels[shift_mask.view(-1)]
            flat_logits = flat_logits[shift_mask.view(-1)]

            if labels_mask.sum() == 0:
                loss_value = 0
            else:
                loss_value = loss_fct(flat_logits, flat_labels)

        proxy_out[f'loss_{seg_num}'] = loss_value
    else:
        proxy_out[f'loss_{seg_num}'] = 0

    segment_keys = ['loss']
    if kwargs.get('output_attentions'):
        segment_keys.append('attentions')
    if kwargs.get('output_hidden_states'):
        segment_keys.append('hidden_states')

    for key, value in cell_out.items():
        if any([sk in key for sk in segment_keys]):
            proxy_out[f'{key}_{seg_num}'] = value

num_segments = len(segments)
out['loss'] = sum([proxy_out[f'loss_{seg_num}'] for seg_num in range(num_segments)]) / num_segments
out['logits'] = torch.cat([cell_out.logits for cell_out in cell_outputs], dim=1)

In [36]:
for k, v in proxy_out.items():
    print(k, v.item() if isinstance(v, torch.Tensor) else v)

loss_0 0
loss_1 9.379081166116521e-05
loss_2 0.0001396716688759625
loss_3 0.00016610838065389544
loss_4 0.0001877214090200141
loss_5 0.00045566188055090606


In [37]:
valid_dataset[0]

{'task': '5 6 3 2 * 7 4 3 4',
 'labels': '5 5 6 0 8 2 0 1',
 'cot': '5 5 5 6 1 + 0 0 6 4 9 0 ( 5 5 1 1 1 1 ) + 0 0 5 9 0 7 0 ( 5 5 6 0 2 8 0 ) + 0 0 0 0 6 4 9 0'}

In [38]:

def compute_accuracy(eval_pred):
    preds = eval_pred.predictions.argmax(axis=-1)[:, :-1]
    labels = eval_pred.label_ids[:, 1:]

    labels_masks = labels > 0
    preds_full = [p[m] for p, m in zip(preds, labels_masks)]
    labels_full = [lab[m] for lab, m in zip(labels, labels_masks)]

    special_tokens = {ans[0], bos[0]}
    acc_cot, acc_ans = [], []
    for lab_tokens, pred_tokens in zip(labels_full, preds_full):
        ans_start_index = max(i for i, x in enumerate(lab_tokens) if x == ans[0])

        pred_cot_tokens = pred_tokens[:ans_start_index].tolist()
        lab_cot_tokens = lab_tokens[:ans_start_index].tolist()

        cot_correct = [p == l for p, l in zip(pred_cot_tokens, lab_cot_tokens) if l not in special_tokens]
        acc_cot.append(all(cot_correct))

        pred_ans_tokens = pred_tokens[ans_start_index:].tolist()
        lab_ans_tokens = lab_tokens[ans_start_index:].tolist()

        ans_correct = [p == l for p, l in zip(pred_ans_tokens, lab_ans_tokens) if l not in special_tokens]
        acc_ans.append(all(ans_correct))

    return {'accuracy_cot': np.mean(acc_cot), 'accuracy_ans': np.mean(acc_ans)}

In [39]:
special_tokens = {ans[0], bos[0]}


In [40]:
tokenizer.decode([p for p, l in zip(pred_ans_tokens, lab_ans_tokens) if l not in special_tokens])

NameError: name 'pred_ans_tokens' is not defined

In [33]:
tokenizer.decode([l for p, l in zip(pred_ans_tokens, lab_ans_tokens) if l not in special_tokens])

'9 1 0 3 0 8 6 0<|endoftext|>'

In [34]:
tokenizer.decode([l for p, l in zip(pred_ans_tokens, lab_ans_tokens)])

'!!!!////9 1 0 3 0 8 6 0<|endoftext|>'

In [35]:
tokenizer.decode([p for p, l in zip(pred_ans_tokens, lab_ans_tokens)])

'!!!! 69 1 0 3 2 8 6 0<|endoftext|>'

In [41]:
eval_pred = Holder()

eval_pred.predictions = out['logits']
eval_pred.label_ids = collated['labels']

In [42]:
acc = compute_accuracy(eval_pred)
print(acc)

{'accuracy_cot': 1.0, 'accuracy_ans': 1.0}


In [58]:
preds = eval_pred.predictions.argmax(axis=-1)[:, :-1]
labels = eval_pred.label_ids[:, 1:]

labels_masks = labels > 0
preds_full = [p[m] for p, m in zip(preds, labels_masks)]
labels_full = [lab[m] for lab, m in zip(labels, labels_masks)]

acc_cot, acc_ans = [], []
correct_samples = []
for lab_tokens, pred_tokens in zip(labels_full, preds_full):
    ans_start_index = max(i for i, x in enumerate(lab_tokens) if x == ans[0])

    pred_cot_tokens = pred_tokens[:ans_start_index].tolist()
    lab_cot_tokens = lab_tokens[:ans_start_index].tolist()

    cot_correct = [p == l for p, l in zip(pred_cot_tokens, lab_cot_tokens) if l != bos[0]]
    acc_cot.append(all(cot_correct))

    pred_ans_tokens = pred_tokens[ans_start_index:].tolist()
    lab_ans_tokens = lab_tokens[ans_start_index:].tolist()

    # ans_correct = [p == l for p, l in zip(pred_ans_tokens, lab_ans_tokens)]
    ans_correct = [p == l for p, l in zip(pred_ans_tokens, lab_ans_tokens) if l not in special_tokens]
    acc_ans.append(all(ans_correct))
    if all(ans_correct):
        correct_samples.append(lab_tokens)
    # break


In [59]:
tokenizer.batch_decode(correct_samples)

['////1 9 7 9 1????////0 0 0 0 0 0 ( 1 9 7 9 1 0 )????////0 0 9 9 1 2 0 ( 1 9 6 9 3 2 0 )????////0 0 0 3 9 3 5 1!!!!////1 9 6 2 3 6 5 1<|endoftext|>',
 '////4 7 3 4 0????////0 2 2 1 3 1 ( 4 9 5 5 3 1 )????////0 0 8 4 7 8 0 ( 4 9 3 0 1 0 1 )????////0 0 0 8 4 7 8 0!!!!////4 9 3 8 5 7 9 0<|endoftext|>',
 '////0 0 2 6 2????////0 0 5 5 6 0 ( 0 0 7 1 9 0 )????////0 0 0 0 2 6 2 ( 0 0 7 1 1 7 2 )????////0 0 0 0 0 2 6 2!!!!////0 0 7 1 1 9 8 2<|endoftext|>',
 '////9 3 1 1 4????////0 2 4 1 9 0 ( 9 5 5 2 3 1 )????////0 0 3 1 7 3 1 ( 9 5 8 3 0 5 1 )????////0 0 0 5 5 8 2 2!!!!////9 5 8 8 5 3 4 2<|endoftext|>',
 '////5 4 9 7 4????////0 2 1 7 6 7 ( 5 6 0 5 1 8 )????////0 0 6 5 3 8 3 ( 5 6 6 0 5 6 4 )????////0 0 0 2 1 7 6 7!!!!////5 6 6 2 6 3 1 8<|endoftext|>',
 '////8 4 6 3 5????////0 2 1 4 3 1 ( 8 6 7 7 8 1 )????////0 0 8 1 1 0 2 ( 8 6 5 9 9 1 2 )????////0 0 0 4 5 3 0 6!!!!////8 6 5 3 5 5 2 6<|endoftext|>',
 '////6 1 2 9 0????////0 0 2 5 1 1 ( 6 1 4 4 2 1 )????////0 0 0 0 0 0 0 ( 6 1 4 4 2 1 0 )????/

In [60]:
tokenizer.batch_decode(labels_full[:5])

['////5 5 5 6 1????////0 0 6 4 9 0 ( 5 5 1 1 1 1 )????////0 0 5 9 0 7 0 ( 5 5 6 0 2 8 0 )????////0 0 0 0 6 4 9 0!!!!////5 5 6 0 8 2 0 1<|endoftext|>',
 '////6 7 1 1 3????////0 4 8 7 0 2 ( 6 1 0 9 3 2 )????////0 0 4 8 7 0 2 ( 6 1 4 7 1 3 2 )????////0 0 0 2 7 3 6 3!!!!////6 1 4 9 8 6 8 3<|endoftext|>',
 '////8 0 0 5 7????////0 4 8 3 4 8 ( 8 4 8 8 1 9 )????////0 0 6 7 3 9 0 ( 8 4 4 6 5 8 1 )????////0 0 0 2 3 6 5 6!!!!////8 4 4 8 8 4 7 6<|endoftext|>',
 '////9 0 9 2 1????////0 2 1 2 7 1 ( 9 2 0 5 8 1 )????////0 0 8 1 8 5 2 ( 9 2 8 6 6 7 2 )????////0 0 0 5 1 5 1 2!!!!////9 2 8 1 8 2 4 2<|endoftext|>',
 '////0 4 6 8 5????////0 0 5 6 6 3 ( 0 4 1 5 2 4 )????////0 0 0 8 9 3 4 ( 0 4 1 3 2 8 4 )????////0 0 0 0 5 6 6 3!!!!////0 4 1 3 7 4 1 4<|endoftext|>']

In [61]:
tokenizer.batch_decode(preds_full[:5])

[' ty5 5 5 6 1????????0 0 6 4 9 0 ( 5 5 1 1 1 1 )????????0 0 5 9 0 7 0 ( 5 5 6 0 2 8 0 )????????0 0 0 0 6 4 9 0!!!! 05 5 6 0 1 2 0 1<|endoftext|>',
 ' deposit6 7 1 1 3????????0 4 8 7 0 2 ( 6 1 0 9 3 2 )????????0 0 4 8 7 0 2 ( 6 1 4 7 1 3 2 )????????0 0 0 2 7 3 6 3!!!! 26 1 4 9 5 6 8 3<|endoftext|>',
 ' Bild8 0 0 5 7????????0 4 8 3 4 8 ( 8 4 8 8 1 9 )????????0 0 6 7 3 9 0 ( 8 4 4 6 5 8 1 )????????0 0 0 2 3 6 5 6!!!! 28 4 4 8 5 4 7 6<|endoftext|>',
 ' Hein9 0 9 2 1????????0 2 1 2 7 1 ( 9 2 0 5 8 1 )????????0 0 8 1 8 5 2 ( 9 2 8 6 6 7 2 )????????0 0 0 5 1 5 1 2!!!! 59 2 8 1 3 2 4 2<|endoftext|>',
 ' fragrance0 4 6 8 5????????0 0 5 6 6 3 ( 0 4 1 5 2 4 )????????0 0 0 8 9 3 4 ( 0 4 1 3 2 8 4 )????????0 0 0 0 5 6 6 3!!!! 00 4 1 3 9 4 1 4<|endoftext|>']

In [36]:
tokenizer.decode(labels_full[0])

'////5 5 5 6 1????////0 0 6 4 9 0 ( 5 5 1 1 1 1 )????////0 0 5 9 0 7 0 ( 5 5 6 0 2 8 0 )????////0 0 0 0 6 4 9 0!!!!////5 5 6 0 8 2 0 1<|endoftext|>'

In [37]:
tokenizer.decode(lab_tokens)

'////9 9 0 9 0????////0 2 3 1 2 1 ( 9 1 4 0 3 1 )????////0 0 6 6 0 6 0 ( 9 1 0 7 3 7 0 )????////0 0 0 6 6 0 6 0!!!!////9 1 0 3 0 8 6 0<|endoftext|>'

In [24]:
tokenizer.decode(ans[0])

'!!!!'

In [25]:
ans_start_index

50

In [38]:
tokenizer.batch_decode(preds_full[:5])

[' ty5 5 5 6 1????????0 0 6 4 9 0 ( 5 5 1 1 1 1 )????????0 0 5 9 0 7 0 ( 5 5 6 0 2 8 0 )????????0 0 0 0 6 4 9 0!!!! 05 5 6 0 1 2 0 1<|endoftext|>',
 ' deposit6 7 1 1 3????????0 4 8 7 0 2 ( 6 1 0 9 3 2 )????????0 0 4 8 7 0 2 ( 6 1 4 7 1 3 2 )????????0 0 0 2 7 3 6 3!!!! 26 1 4 9 5 6 8 3<|endoftext|>',
 ' Bild8 0 0 5 7????????0 4 8 3 4 8 ( 8 4 8 8 1 9 )????????0 0 6 7 3 9 0 ( 8 4 4 6 5 8 1 )????????0 0 0 2 3 6 5 6!!!! 28 4 4 8 5 4 7 6<|endoftext|>',
 ' Hein9 0 9 2 1????????0 2 1 2 7 1 ( 9 2 0 5 8 1 )????????0 0 8 1 8 5 2 ( 9 2 8 6 6 7 2 )????????0 0 0 5 1 5 1 2!!!! 59 2 8 1 3 2 4 2<|endoftext|>',
 ' fragrance0 4 6 8 5????????0 0 5 6 6 3 ( 0 4 1 5 2 4 )????????0 0 0 8 9 3 4 ( 0 4 1 3 2 8 4 )????????0 0 0 0 5 6 6 3!!!! 00 4 1 3 9 4 1 4<|endoftext|>']

In [39]:
tokenizer.batch_decode(labels_full[:5])

['////5 5 5 6 1????////0 0 6 4 9 0 ( 5 5 1 1 1 1 )????////0 0 5 9 0 7 0 ( 5 5 6 0 2 8 0 )????////0 0 0 0 6 4 9 0!!!!////5 5 6 0 8 2 0 1<|endoftext|>',
 '////6 7 1 1 3????////0 4 8 7 0 2 ( 6 1 0 9 3 2 )????////0 0 4 8 7 0 2 ( 6 1 4 7 1 3 2 )????////0 0 0 2 7 3 6 3!!!!////6 1 4 9 8 6 8 3<|endoftext|>',
 '////8 0 0 5 7????////0 4 8 3 4 8 ( 8 4 8 8 1 9 )????////0 0 6 7 3 9 0 ( 8 4 4 6 5 8 1 )????////0 0 0 2 3 6 5 6!!!!////8 4 4 8 8 4 7 6<|endoftext|>',
 '////9 0 9 2 1????////0 2 1 2 7 1 ( 9 2 0 5 8 1 )????////0 0 8 1 8 5 2 ( 9 2 8 6 6 7 2 )????////0 0 0 5 1 5 1 2!!!!////9 2 8 1 8 2 4 2<|endoftext|>',
 '////0 4 6 8 5????////0 0 5 6 6 3 ( 0 4 1 5 2 4 )????////0 0 0 8 9 3 4 ( 0 4 1 3 2 8 4 )????////0 0 0 0 5 6 6 3!!!!////0 4 1 3 7 4 1 4<|endoftext|>']

In [65]:
tokenizer.batch_decode([s[-9:] for s in preds_full[:5]])

['5 5 6 0 1 2 0 1<|endoftext|>',
 '6 1 4 9 5 6 8 3<|endoftext|>',
 '8 4 4 8 5 4 7 6<|endoftext|>',
 '9 2 8 1 3 2 4 2<|endoftext|>',
 '0 4 1 3 9 4 1 4<|endoftext|>']

In [66]:
tokenizer.batch_decode([s[-9:] for s in labels_full[:5]])

['5 5 6 0 8 2 0 1<|endoftext|>',
 '6 1 4 9 8 6 8 3<|endoftext|>',
 '8 4 4 8 8 4 7 6<|endoftext|>',
 '9 2 8 1 8 2 4 2<|endoftext|>',
 '0 4 1 3 7 4 1 4<|endoftext|>']

In [40]:
tokenizer.decode([p for p, l in zip(pred_cot_tokens, lab_cot_tokens) if l != bos[0]])

'9 9 0 9 0????0 2 3 1 2 1 ( 9 1 4 0 3 1 )????0 0 6 6 0 6 0 ( 9 1 0 7 3 7 0 )????0 0 0 6 6 0 6 0'

In [41]:
tokenizer.decode([l for p, l in zip(pred_cot_tokens, lab_cot_tokens) if l != bos[0]])

'9 9 0 9 0????0 2 3 1 2 1 ( 9 1 4 0 3 1 )????0 0 6 6 0 6 0 ( 9 1 0 7 3 7 0 )????0 0 0 6 6 0 6 0'

In [167]:
cot_correct

[True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True]

In [16]:
# np.mean([value.item() if isinstance(value, torch.Tensor) else value  for value in proxy_out.values()])

In [17]:
cell_outputs[0].keys()

odict_keys(['logits', 'hidden_states'])

In [18]:
si = 1

In [19]:
# inputs
for seg in segments[:]:
    print(tokenizer.decode(seg['input_ids'][si]))

////6 9 1 5 * 6 4 4 7????
////6 7 1 1 3????
////0 4 8 7 0 2 ( 6 1 0 9 3 2 )????
////0 0 4 8 7 0 2 ( 6 1 4 7 1 3 2 )????
////0 0 0 2 7 3 6 3!!!!
////6 1 4 9 8 6 8 3<|endoftext|>


In [20]:
# predictions
[tokenizer.decode(o.logits.argmax(dim=-1)[si]) for o in cell_outputs]

['robees buttons 2 4 5- 2 dunk wellnesspard',
 '6 7 1 1 3????????',
 '0 4 8 7 0 2 ( 6 1 0 9 3 2 )???? 9',
 '0 0 4 8 7 0 2 ( 6 1 4 7 1 3 2 )???? 7',
 '0 0 0 2 7 3 6 3!!!!!!!!',
 '6 1 4 9 1 6 8 3<|endoftext|><|endoftext|>']

In [ ]:
def compute_accuracy(eval_pred):
    preds = eval_pred.predictions.argmax(axis=-1)[:, :-1]
    labels = eval_pred.label_ids[:, 1:]

    labels_masks = labels > 0
    preds_full = [p[m] for p, m in zip(preds, labels_masks)]
    labels_full = [lab[m] for lab, m in zip(labels, labels_masks)]

    special_tokens = {ans[0], bos[0]}
    acc_cot, acc_ans = [], []
    for lab_tokens, pred_tokens in zip(labels_full, preds_full):
        ans_start_index = max(i for i, x in enumerate(lab_tokens) if x == ans[0])

        pred_cot_tokens = pred_tokens[:ans_start_index].tolist()
        lab_cot_tokens = lab_tokens[:ans_start_index].tolist()

        cot_correct = [p == l for p, l in zip(pred_cot_tokens, lab_cot_tokens) if l not in special_tokens]
        acc_cot.append(all(cot_correct))

        pred_ans_tokens = pred_tokens[ans_start_index:].tolist()
        lab_ans_tokens = lab_tokens[ans_start_index:].tolist()

        ans_correct = [p == l for p, l in zip(pred_ans_tokens, lab_ans_tokens)]
        acc_ans.append(all(ans_correct))

    return {'accuracy_cot': np.mean(acc_cot), 'accuracy_ans': np.mean(acc_ans)}

tensor([[ -100,  -100,  -100,  ...,   657,   352, 50256],
        [ -100,  -100,  -100,  ...,   807,   513, 50256],
        [ -100,  -100,  -100,  ...,   767,   718, 50256],
        ...,
        [ -100,  -100,  -100,  ...,   807,   657, 50256],
        [ -100,  -100,  -100,  ...,   642,   352, 50256],
        [ -100,  -100,  -100,  ...,   604,   807, 50256]])

In [41]:
preds = out['logits'].argmax(axis=-1)[:, :-1]
labels = collated['labels'][:, 1:]
labels_masks = labels > 0

In [42]:
preds_full = [p[m] for p, m in zip(preds, labels_masks)]
labels_full = [lab[m] for lab, m in zip(labels, labels_masks)]

print(len(preds_full), len(labels_full))

preds_full_text = tokenizer.batch_decode(preds_full, add_special_tokens=True)
labels_full_text = tokenizer.batch_decode(labels_full, add_special_tokens=True)

16 16


In [54]:
think

[9805]

In [ ]:
preds_full[0]

AttributeError: 'Tensor' object has no attribute 'find'

In [ ]:
def get_ans(tensor):
    tokens = tensor.tolist()
    
    try:
        start_index = max(i for i, x in enumerate(tokens) if x == ans[0])
    except ValueError:
        return []
    
    tokens = tokens[start_index + 1:]
    
    if tokens and tokens[0] == bos[0]:
        tokens = tokens[1:]
    
    return torch.tensor(tokens)

def get_cot(tensor)

In [109]:
tokens = labels_full[0].tolist()

ans_start_index = max(i for i, x in enumerate(tokens) if x == ans[0])



In [145]:
lab_tokens = labels_full[0]
pred_tokens = preds_full[0]

special_tokens = {ans[0], bos[0]}
pred_cot_tokens = pred_tokens[:ans_start_index].tolist()
lab_cot_tokens = lab_tokens[:ans_start_index].tolist()

cot_correct = [p == l for p, l in zip(pred_cot_tokens, lab_cot_tokens) if l not in special_tokens]

pred_ans_tokens = pred_tokens[ans_start_index:].tolist()
lab_ans_tokens = lab_tokens[ans_start_index:].tolist()

ans_correct = [p == l for p, l in zip(pred_ans_tokens, lab_ans_tokens)]

In [147]:
np.mean(cot_correct)

1.0

In [148]:
np.mean(ans_correct)

0.8181818181818182

In [142]:
tokenizer.decode(pred_tokens)

' 45 5 5 6 1????????0 0 6 4 9 0 ( 5 5 1 1 1 1 )????????0 0 5 9 0 7 0 ( 5 5 6 0 2 8 0 )???? 00 0 0 0 6 4 9 0!!!!!!!!5 5 6 0 7 2 0 1<|endoftext|>'

In [ ]:
# preds
tokenizer.decode([p for p, l in zip(pred_cot_tokens, lab_cot_tokens) if l not in special_tokens])

'5 5 5 6 1????0 0 6 4 9 0 ( 5 5 1 1 1 1 )????0 0 5 9 0 7 0 ( 5 5 6 0 2 8 0 )????0 0 0 0 6 4 9 0'

In [ ]:
# labels
tokenizer.decode([l for p, l in zip(pred_cot_tokens, lab_cot_tokens) if l not in special_tokens])

'5 5 5 6 1????0 0 6 4 9 0 ( 5 5 1 1 1 1 )????0 0 5 9 0 7 0 ( 5 5 6 0 2 8 0 )????0 0 0 0 6 4 9 0'

In [ ]:
[l for l in lab_cot_tokens if l not in special_tokens]

[9805, 9805, 9805]

In [131]:
think[0]

9805

In [132]:
pred_cot_tokens.shape

AttributeError: 'list' object has no attribute 'shape'

In [133]:
len(cot_correct)

47

In [107]:
# tokens

In [98]:
len(get_ans(preds_full[0])), len(get_ans(labels_full[0])), 

(9, 9)

In [99]:
tokenizer.decode(get_ans(preds_full[0]))

'5 5 6 0 7 2 0 1<|endoftext|>'

In [100]:
tokenizer.decode(get_ans(labels_full[0]))

'5 5 6 0 8 2 0 1<|endoftext|>'

In [69]:
def get_ans(text):
    splits = text.split(ans_text)
    splits = list(filter(len, splits))
    ans = splits[-1].strip()
    if ans.startswith(ans_text):
        ans = ans[len(ans_text):]
    if ans.startswith(bos_text):
        ans = ans[len(bos_text):]
    return ans

In [70]:
get_ans(preds_full_text[0]), get_ans(labels_full_text[0])

('5 5 6 0 7 2 0 1<|endoftext|>', '5 5 6 0 8 2 0 1<|endoftext|>')

In [71]:
preds_full_text[0].split(ans_text)

[' 45 5 5 6 1????????0 0 6 4 9 0 ( 5 5 1 1 1 1 )????????0 0 5 9 0 7 0 ( 5 5 6 0 2 8 0 )???? 00 0 0 0 6 4 9 0',
 '',
 '5 5 6 0 7 2 0 1<|endoftext|>']

In [ ]:
labels_full_text[0].split(ans_text)

['////5 5 5 6 1????////0 0 6 4 9 0 ( 5 5 1 1 1 1 )????////0 0 5 9 0 7 0 ( 5 5 6 0 2 8 0 )????////0 0 0 0 6 4 9 0',
 '////5 5 6 0 8 2 0 1<|endoftext|>']

In [65]:
preds_full_text

[' 45 5 5 6 1????????0 0 6 4 9 0 ( 5 5 1 1 1 1 )????????0 0 5 9 0 7 0 ( 5 5 6 0 2 8 0 )???? 00 0 0 0 6 4 9 0!!!!!!!!5 5 6 0 7 2 0 1<|endoftext|>',
 'pard6 7 1 1 3????????0 4 8 7 0 2 ( 6 1 0 9 3 2 )???? 90 0 4 8 7 0 2 ( 6 1 4 7 1 3 2 )???? 70 0 0 2 7 3 6 3!!!!!!!!6 1 4 9 1 6 8 3<|endoftext|>',
 ' Doe8 0 0 5 7????????0 4 8 3 4 8 ( 8 4 8 8 1 9 )????????0 0 6 7 3 9 0 ( 8 4 4 6 5 8 1 )???? 60 0 0 2 3 6 5 6!!!!!!!!8 4 4 8 1 4 7 6<|endoftext|>',
 ' Gentleman9 0 9 2 1????????0 2 1 2 7 1 ( 9 2 0 5 8 1 )???? 50 0 8 1 8 5 2 ( 9 2 8 6 6 7 2 )???? 60 0 0 5 1 5 1 2!!!!!!!!9 2 8 1 0 2 4 2<|endoftext|>',
 ' drum0 4 6 8 5????????0 0 5 6 6 3 ( 0 4 1 5 2 4 )???? 50 0 0 8 9 3 4 ( 0 4 1 3 2 8 4 )???? 30 0 0 0 5 6 6 3!!!!!!!!0 4 1 3 9 4 1 4<|endoftext|>',
 ' 42 5 2 4 2????????0 9 8 1 8 1 ( 2 4 1 6 0 2 )???? 60 0 4 0 5 8 4 ( 2 4 5 6 5 0 5 )???? 60 0 0 1 4 4 2 4!!!!!!!!2 4 5 7 1 4 7 4<|endoftext|>',
 'hovah8 4 7 9 0????????0 6 6 8 3 4 ( 8 0 4 8 4 4 )???? 80 0 4 7 8 4 0 ( 8 0 8 5 3 9 0 )???? 50 0 0 4 4 2 9 2!!

In [48]:
labels_full_text

['////5 5 5 6 1????////0 0 6 4 9 0 ( 5 5 1 1 1 1 )????////0 0 5 9 0 7 0 ( 5 5 6 0 2 8 0 )????////0 0 0 0 6 4 9 0!!!!////5 5 6 0 8 2 0 1<|endoftext|>',
 '////6 7 1 1 3????////0 4 8 7 0 2 ( 6 1 0 9 3 2 )????////0 0 4 8 7 0 2 ( 6 1 4 7 1 3 2 )????////0 0 0 2 7 3 6 3!!!!////6 1 4 9 8 6 8 3<|endoftext|>',
 '////8 0 0 5 7????////0 4 8 3 4 8 ( 8 4 8 8 1 9 )????////0 0 6 7 3 9 0 ( 8 4 4 6 5 8 1 )????////0 0 0 2 3 6 5 6!!!!////8 4 4 8 8 4 7 6<|endoftext|>',
 '////9 0 9 2 1????////0 2 1 2 7 1 ( 9 2 0 5 8 1 )????////0 0 8 1 8 5 2 ( 9 2 8 6 6 7 2 )????////0 0 0 5 1 5 1 2!!!!////9 2 8 1 8 2 4 2<|endoftext|>',
 '////0 4 6 8 5????////0 0 5 6 6 3 ( 0 4 1 5 2 4 )????////0 0 0 8 9 3 4 ( 0 4 1 3 2 8 4 )????////0 0 0 0 5 6 6 3!!!!////0 4 1 3 7 4 1 4<|endoftext|>',
 '////2 5 2 4 2????////0 9 8 1 8 1 ( 2 4 1 6 0 2 )????////0 0 4 0 5 8 4 ( 2 4 5 6 5 0 5 )????////0 0 0 1 4 4 2 4!!!!////2 4 5 7 9 4 7 4<|endoftext|>',
 '////8 4 7 9 0????////0 6 6 8 3 4 ( 8 0 4 8 4 4 )????////0 0 4 7 8 4 0 ( 8 0 8 5 3 9 0 )????/

In [ ]:

preds_cot = [extract_cot(p) for p in preds_full_text]
preds_ans = [extract_answer(p) for p in preds_full_text]

labels_cot = [extract_cot(lab) for lab in labels_full_text]
labels_ans = [extract_answer(lab) for lab in labels_full_text]

acc_cot = np.mean([c == p for c, p in zip(preds_cot, labels_cot)])
acc_ans = np.mean([c == lab for c, lab in zip(preds_ans, labels_ans)])

In [32]:
preds.shape, labels.shape

(torch.Size([16, 72]), torch.Size([1, 72]))

In [33]:
preds

tensor([[25481,  1666, 13129,  ...,   352, 50256, 50256],
        [25481,   274, 12163,  ...,   513, 50256, 50256],
        [25481,   274,   513,  ...,   718, 50256, 50256],
        ...,
        [25481,   274,   513,  ...,   657, 50256, 50256],
        [25481,   657,   513,  ...,   352, 50256, 50256],
        [25481,    67,   352,  ...,   807, 50256, 50256]])

In [21]:
# tokenizer.decode([segment['labels'][0][0] for segment in segments[1:]])

In [22]:
# import pandas as pd
# res_df = pd.DataFrame(columns=['cpt_path', 'cot', 'acc_cot', 'acc_ans'])

In [83]:
[o.loss for o in cell_outputs]

[None, None, None, None, None, None]

In [ ]:

model.generation_config.pad_token_id = tokenizer.pad_token_id
gen_outputs = [model.generate(inp.reshape(1, -1).to(device), 
                            pad_token_id=tokenizer.eos_token_id,
                            attention_mask=torch.ones_like(inp.reshape(1, -1)).to(device),
                            max_new_tokens=50)[0] for inp in collated['input_ids_generate']]

if args.use_cot:
    gen_outputs = [model.generate(inp.reshape(1, -1).to(device), 
                                    pad_token_id=tokenizer.eos_token_id,
                                    attention_mask=torch.ones_like(inp.reshape(1, -1)).to(device))[0].cpu() for inp in gen_outputs]

labels = collated['labels']
labels_masks = labels > 0

preds_full = [out[len(inp):] for inp, out in zip(collated['input_ids_generate'], gen_outputs)]
# preds_full = [out[len(inp):] for inp, out in zip(collated['input_ids_generate'], gen_outputs_m2)]
labels_full = [lab[m][1:] for lab, m in zip(labels, labels_masks)]

print(len(preds_full), len(labels_full))

preds_full_text = tokenizer.batch_decode(preds_full, add_special_tokens=True)
labels_full_text = tokenizer.batch_decode(labels_full, add_special_tokens=True)

preds_cot = [extract_cot(p) for p in preds_full_text]
preds_ans = [extract_answer(p) for p in preds_full_text]

labels_cot = [extract_cot(lab) for lab in labels_full_text]
labels_ans = [extract_answer(lab) for lab in labels_full_text]

acc_cot = np.mean([c == p for c, p in zip(preds_cot, labels_cot)])
acc_ans = np.mean([c == lab for c, lab in zip(preds_ans, labels_ans)])

data = {"inputs": collated['input_ids_generate'],
        "preds_full_text": preds_full_text, "labels_full_text": labels_full_text,
        "preds_cot": preds_cot, "labels_cot": labels_cot,
        "preds_ans": preds_ans, "labels_ans": labels_ans}

print(f"Accuracy COT: {acc_cot}")
print(f"Accuracy Answer: {acc_ans}")

In [77]:
args.use_cot = False

checkpoints = [
    "/workspace-SR006.nfs2/Bulatov_A/rmt/runs/gsm8k/gpt2/SEGM_1x1024_1024_LR3e-04/checkpoint-16500/pytorch_model.bin",
    "/workspace-SR006.nfs2/Bulatov_A/rmt/runs/gsm8k/gpt2/SEGM_1x1024_1024_LR1e-03/checkpoint-24500/pytorch_model.bin",
]

for cpt_path in checkpoints:
    print(cpt_path)
    acc_cot, acc_ans = evaluate_model_on_dataset(cpt_path, valid_dataset)
    res_df.loc[len(res_df)] = [cpt_path.split('/')[-3], args.use_cot, 0, acc_cot]

/workspace-SR006.nfs2/Bulatov_A/rmt/runs/gsm8k/gpt2/SEGM_1x1024_1024_LR3e-04/checkpoint-16500/pytorch_model.bin
500 500
Accuracy COT: 0.172
Accuracy Answer: 1.0
/workspace-SR006.nfs2/Bulatov_A/rmt/runs/gsm8k/gpt2/SEGM_1x1024_1024_LR1e-03/checkpoint-24500/pytorch_model.bin
500 500
Accuracy COT: 0.148
Accuracy Answer: 1.0


In [78]:
res_df

,cpt_path,cot,acc_cot,acc_ans
0,SEGM_1x1024_1024_LR1e-03-cot,True,0.172,0.450
1,SEGM_1x1024_1024_LR3e-04-cot,True,0.156,0.392
2,SEGM_1x1024_1024_LR3e-04,False,0.000,0.172
3,SEGM_1x1024_1024_LR1e-03,False,0.000,0.148


In [60]:
# model.load_state_dict(torch.load(checkpoint_path), strict=False)
model.generation_config.pad_token_id = tokenizer.pad_token_id
model.to(device)
    
collated = collate_fn([sample for sample in valid_dataset])
model.generation_config.pad_token_id = tokenizer.pad_token_id
gen_outputs = [model.generate(inp.reshape(1, -1).to(device), 
                            pad_token_id=tokenizer.eos_token_id,
                            attention_mask=torch.ones_like(inp.reshape(1, -1)).to(device),
                            max_new_tokens=50)[0] for inp in collated['input_ids_generate']]

gen_outputs_m2 = [model.generate(inp.reshape(1, -1).to(device), 
                                    pad_token_id=tokenizer.eos_token_id,
                                    attention_mask=torch.ones_like(inp.reshape(1, -1)).to(device))[0].cpu() for inp in gen_outputs]

labels = collated['labels']
labels_masks = labels > 0

preds_full = [out[len(inp):] for inp, out in zip(collated['input_ids_generate'], gen_outputs_m2)]
labels_full = [lab[m][1:] for lab, m in zip(labels, labels_masks)]

print(len(preds_full), len(labels_full))

preds_full_text = tokenizer.batch_decode(preds_full, add_special_tokens=True)
labels_full_text = tokenizer.batch_decode(labels_full, add_special_tokens=True)

preds_cot = [extract_cot(p) for p in preds_full_text]
preds_ans = [extract_answer(p) for p in preds_full_text]

labels_cot = [extract_cot(lab) for lab in labels_full_text]
labels_ans = [extract_answer(lab) for lab in labels_full_text]

acc_cot = np.mean([c == p for c, p in zip(preds_cot, labels_cot)])
acc_ans = np.mean([c == lab for c, lab in zip(preds_ans, labels_ans)])

print(f"Accuracy COT: {acc_cot}")
print(f"Accuracy Answer: {acc_ans}")

500 500
Accuracy COT: 0.148
Accuracy Answer: 1.0


In [ ]:
def extract_cot(text):
        try:
                return text[:text.index(ans_text)]
        except ValueError:
                return ''

def extract_answer(text):
        try:
                return text.split(ans_text)[1]
        except IndexError:
                return ''

In [66]:
preds_full_text

['600<|endoftext|><|endoftext|>',
 '7.5<|endoftext|><|endoftext|>',
 '1400<|endoftext|><|endoftext|>',
 '15<|endoftext|><|endoftext|>',
 '240<|endoftext|><|endoftext|>',
 '20<|endoftext|><|endoftext|>',
 '160<|endoftext|><|endoftext|>',
 '1.5<|endoftext|><|endoftext|>',
 '21.75<|endoftext|><|endoftext|>',
 '21.5<|endoftext|><|endoftext|>',
 '14<|endoftext|><|endoftext|>',
 '88<|endoftext|><|endoftext|>',
 '20<|endoftext|><|endoftext|>',
 '130<|endoftext|><|endoftext|>',
 '16.67<|endoftext|><|endoftext|>',
 '33.33<|endoftext|><|endoftext|>',
 '25<|endoftext|><|endoftext|>',
 '-4<|endoftext|><|endoftext|>',
 '90<|endoftext|><|endoftext|>',
 '46<|endoftext|><|endoftext|>',
 '715<|endoftext|><|endoftext|>',
 '5<|endoftext|><|endoftext|>',
 '10<|endoftext|><|endoftext|>',
 '1<|endoftext|><|endoftext|>',
 '12000<|endoftext|><|endoftext|>',
 '160<|endoftext|><|endoftext|>',
 '5<|endoftext|><|endoftext|>',
 '45<|endoftext|><|endoftext|>',
 '74<|endoftext|><|endoftext|>',
 '460<|endoftext|><|en

In [62]:
preds_cot[:10]

['600', '7.5', '1400', '15', '240', '20', '160', '1.5', '21.75', '21.5']

In [63]:
labels_cot[:10]

['300', '10', '1400', '15', '240', '20', '10', '2', '25', '25']

In [64]:
preds_ans[:10]


['', '', '', '', '', '', '', '', '', '']

In [65]:
labels_ans[:10]

['', '', '', '', '', '', '', '', '', '']

In [13]:

labels = collated['labels'][:, 1:]
labels_masks = labels > 0

preds_full = [out[len(inp):] for inp, out in zip(collated['input_ids_generate'], gen_outputs_m2)]
labels_full = [lab[m][1:] for lab, m in zip(labels, labels_masks)]

print(len(preds_full), len(labels_full))

preds_full_text = tokenizer.batch_decode(preds_full, add_special_tokens=True)
labels_full_text = tokenizer.batch_decode(labels_full, add_special_tokens=True)

preds_cot = [extract_cot(p) for p in preds_full_text]
preds_ans = [extract_answer(p) for p in preds_full_text]

labels_cot = [extract_cot(lab) for lab in labels_full_text]
labels_ans = [extract_answer(lab) for lab in labels_full_text]

acc_cot = np.mean([c == p for c, p in zip(preds_cot, labels_cot)])
acc_ans = np.mean([c == lab for c, lab in zip(preds_ans, labels_ans)])

print(f"Accuracy COT: {acc_cot}")
print(f"Accuracy Answer: {acc_ans}")

5 5
Accuracy COT: 0.6
Accuracy Answer: 1.0


In [5]:
dataset = 'booydar/gsm8k'
train_dataset = datasets.load_dataset(dataset, split='train')
valid_dataset = datasets.load_dataset(dataset, split='valid')

# cot
args.use_cot = True

checkpoints = [
    "/workspace-SR006.nfs2/Bulatov_A/rmt/runs/gsm8k/gpt2/SEGM_1x1024_1024_LR1e-03-cot/checkpoint-23500/pytorch_model.bin",
    "/workspace-SR006.nfs2/Bulatov_A/rmt/runs/gsm8k/gpt2/SEGM_1x1024_1024_LR1e-05-cot/checkpoint-5000/pytorch_model.bin",
    "/workspace-SR006.nfs2/Bulatov_A/rmt/runs/gsm8k/gpt2/SEGM_1x1024_1024_LR3e-04-cot/checkpoint-17500/pytorch_model.bin",
]

In [35]:
# cpt_path = "/workspace-SR006.nfs2/Bulatov_A/rmt/runs/test/4_by_4_mult/Llama-3.2-1B-Instruct/smol:qa1-5-1:9/SEGM_1x1024_1024_64_LR3e-04-lora-mnc-distill__short/checkpoint-5000/pytorch_model.bin"
# cpt_path = "/workspace-SR006.nfs2/Bulatov_A/rmt/runs/test/4_by_4_mult/gpt2/SEGM_1x1024_1024_LR3e-04/checkpoint-25000/pytorch_model.bin"
# cpt_path = "/workspace-SR006.nfs2/Bulatov_A/rmt/runs/gsm8k/gpt2/SEGM_1x1024_1024_LR3e-04/checkpoint-16500/pytorch_model.bin"
cpt_path = "/workspace-SR006.nfs2/Bulatov_A/rmt/runs/gsm8k/gpt2/SEGM_1x1024_1024_LR1e-03-cot/checkpoint-23500/pytorch_model.bin"
model.load_state_dict(torch.load(cpt_path), strict=False)

<All keys matched successfully>

In [ ]:
# dataset_dir = "/workspace-SR006.nfs2/Bulatov_A/rmt/data/implicit_chain_of_thought/4_by_4_mult"

# train_path = os.path.join(dataset_dir, "train")
# valid_path = os.path.join(dataset_dir, "valid")
# train_dataset = datasets.load_from_disk(train_path)
# valid_dataset = datasets.load_from_disk(valid_path)



Generating train_no_aug split: 100%|██████████| 6973/6973 [00:00<00:00, 436918.42 examples/s]


In [77]:
class Holder:
    def __init__(self):
        pass
args = Holder()
# args.use_cot = False
args.use_cot = True
args.num_mem_tokens = None

In [118]:

id_pad_value = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else tokenizer.eos_token_id
think = ans = tokenizer.bos_token_id
eos = tokenizer.eos_token_id

def collate_fn(batch):
    input_ids, input_ids_generate, labels, labels_mask, attention_mask = [], [], [], [], []
    for sample in batch:
        task, lab, cot = sample['task'], sample['labels'], sample['cot']
        task_tokens = tokenizer.encode(task, add_special_tokens=False)
        labels_tokens = tokenizer.encode(lab, add_special_tokens=False)
        cot_tokens = tokenizer.encode(cot, add_special_tokens=False)

        if args.use_cot:
            full_input = task_tokens + [think] + cot_tokens + [ans] + labels_tokens + [eos]
            gen_input = task_tokens + [think]
        else:
            full_input = task_tokens + [ans] + labels_tokens + [eos]
            gen_input = task_tokens + [ans]
        
        inp_ids = torch.tensor(full_input)
        input_ids.append(inp_ids)
        input_ids_generate.append(torch.tensor(gen_input))


        lab = torch.tensor(full_input)
        lab[:len(task_tokens)] = -100
        labels.append(lab)

        lab_mask = torch.ones_like(inp_ids)
        lab_mask[:len(task_tokens)] = 0
        labels_mask.append(lab_mask)
        attention_mask.append(torch.ones_like(inp_ids))

    input_ids = pad_sequence(input_ids, padding_value=id_pad_value, batch_first=True)
    # input_ids_generate = pad_sequence(input_ids_generate, padding_value=id_pad_value, batch_first=True)
    attention_mask = pad_sequence(attention_mask, padding_value=0, batch_first=True)
    labels = pad_sequence(labels, padding_value=id_pad_value, batch_first=True)
    labels_mask = pad_sequence(labels_mask, padding_value=0, batch_first=True)

    collated = {'input_ids': input_ids,
                'input_ids_generate': input_ids_generate,
                'labels': labels,
                'attention_mask': attention_mask,
                }
    if args.num_mem_tokens is not None:
        # add labels mask only for RMT, ARMT
        collated['labels_mask'] = labels_mask.bool()
    return collated

In [167]:
think_text = tokenizer.decode(think)
ans_text = tokenizer.decode(ans)

def extract_cot(text):
        try:
                return text[:text.index(ans_text)]
        except ValueError:
                return ''

def extract_answer(text):
        try:
                return text.split(ans_text)[1]
        except IndexError:
                return ''
                
def compute_accuracy(eval_pred):
        preds = eval_pred.predictions.argmax(axis=-1)[:, 1:-1]
        labels = eval_pred.label_ids[:, 2:]

        labels_masks = labels > 0
        preds_full = [p[m] for p, m in zip(preds, labels_masks)]
        labels_full = [lab[m] for lab, m in zip(labels, labels_masks)]

        print(len(preds_full), len(labels_full))

        preds_full_text = tokenizer.batch_decode(preds_full, add_special_tokens=True)
        labels_full_text = tokenizer.batch_decode(labels_full, add_special_tokens=True)

        preds_cot = [extract_cot(p) for p in preds_full_text]
        preds_ans = [extract_answer(p) for p in preds_full_text]

        labels_cot = [extract_cot(lab) for lab in labels_full_text]
        labels_ans = [extract_answer(lab) for lab in labels_full_text]

        acc_cot = np.mean([c == p for c, p in zip(preds_cot, labels_cot)])
        acc_ans = np.mean([c == lab for c, lab in zip(preds_ans, labels_ans)])

        return {'accuracy_cot': acc_cot, 'accuracy_ans': acc_ans}

In [119]:
batch = [valid_dataset[i] for i in range(10)]
collated = collate_fn(batch)

out = model(**collated)
print(out.keys())

odict_keys(['loss', 'logits', 'past_key_values'])



### no teacher forcing


In [135]:
model.generation_config.pad_token_id = tokenizer.pad_token_id

In [ ]:
model.generation_config.pad_token_id = tokenizer.pad_token_id
gen_outputs = [model.generate(inp.reshape(1, -1), 
                              attention_mask=torch.ones_like(inp.reshape(1, -1)),
                              max_new_tokens=50)[0] for inp in collated['input_ids_generate']]

gen_outputs_m2 = [model.generate(inp.reshape(1, -1), attention_mask=torch.ones_like(inp.reshape(1, -1)))[0] for inp in gen_outputs]

In [233]:
# gen_outputs_m2 = [out[len(inp):] for inp, out in zip(collated['input_ids_generate'], gen_outputs_m2)]

In [234]:
tokenizer.batch_decode(gen_outputs_m2)

['<<4-2=2>> <<2/.5=4>> <<12/4=3>> <<100*3=300>><|endoftext|>300<|endoftext|>',
 '<<1.5*2=3>> <<3+2.5=5.5>> <<1.5+3+5.5=10>><|endoftext|>10<|endoftext|>',
 '<<2000*30/100=600>> <<2000-600=1400>><|endoftext|>1400<|endoftext|>',
 '<<21/7=3>> <<5*3=15>><|endoftext|>15<|endoftext|>',
 '<<200*3=600>> <<600*0.4=240>><|endoftext|>240<|endoftext|>',
 '<<1/2*100=50>> <<3/5*50=30>> <<50+30=80>> <<100-80=20>><|endoftext|>20<|endoftext|>',
 '<<40*2=80>> <<80*2=160>><|endoftext|>160<|endoftext|>',
 '<<12*2=24>> <<4*1=4>> <<4*3=12>> <<24+4+12=40>> <<40/4=10>><|endoftext|>10<|endoftext|>',
 '<<2*2.25=4.50>> <<4.50+3.50+4+3.50=15.00>> <<2*2.50=5.00>> <<15+5+3.50=23.50>><|endoftext|>',
 '<<28/4=7>> <<3*7+1=22>><|endoftext|>22<|endoftext|>']

In [242]:
preds = gen_outputs_m2
labels_masks = labels > 0

preds_full = gen_outputs_m2 #[out[len(inp):] for inp, out in zip(collated['input_ids_generate'], gen_outputs_m2)]
labels_full = [lab[m][1:] for lab, m in zip(labels, labels_masks)]

print(len(preds_full), len(labels_full))

preds_full_text = tokenizer.batch_decode(preds_full, add_special_tokens=True)
labels_full_text = tokenizer.batch_decode(labels_full, add_special_tokens=True)

preds_cot = [extract_cot(p) for p in preds_full_text]
preds_ans = [extract_answer(p) for p in preds_full_text]

labels_cot = [extract_cot(lab) for lab in labels_full_text]
labels_ans = [extract_answer(lab) for lab in labels_full_text]

acc_cot = np.mean([c == p for c, p in zip(preds_cot, labels_cot)])
acc_ans = np.mean([c == lab for c, lab in zip(preds_ans, labels_ans)])

print(f"Accuracy COT: {acc_cot}")
print(f"Accuracy Answer: {acc_ans}")

10 10
Accuracy COT: 0.3
Accuracy Answer: 0.6


In [243]:
preds_full_text

['<<4-2=2>> <<2/.5=4>> <<12/4=3>> <<100*3=300>><|endoftext|>300<|endoftext|>',
 '<<1.5*2=3>> <<3+2.5=5.5>> <<1.5+3+5.5=10>><|endoftext|>10<|endoftext|>',
 '<<2000*30/100=600>> <<2000-600=1400>><|endoftext|>1400<|endoftext|>',
 '<<21/7=3>> <<5*3=15>><|endoftext|>15<|endoftext|>',
 '<<200*3=600>> <<600*0.4=240>><|endoftext|>240<|endoftext|>',
 '<<1/2*100=50>> <<3/5*50=30>> <<50+30=80>> <<100-80=20>><|endoftext|>20<|endoftext|>',
 '<<40*2=80>> <<80*2=160>><|endoftext|>160<|endoftext|>',
 '<<12*2=24>> <<4*1=4>> <<4*3=12>> <<24+4+12=40>> <<40/4=10>><|endoftext|>10<|endoftext|>',
 '<<2*2.25=4.50>> <<4.50+3.50+4+3.50=15.00>> <<2*2.50=5.00>> <<15+5+3.50=23.50>><|endoftext|>',
 '<<28/4=7>> <<3*7+1=22>><|endoftext|>22<|endoftext|>']

In [244]:
labels_full_text

['<<4-2=2>> <<2/.5=4>> <<12/4=3>> <<100*3=300>><|endoftext|>300<|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><

In [245]:
preds_cot

['<<4-2=2>> <<2/.5=4>> <<12/4=3>> <<100*3=300>>',
 '<<1.5*2=3>> <<3+2.5=5.5>> <<1.5+3+5.5=10>>',
 '<<2000*30/100=600>> <<2000-600=1400>>',
 '<<21/7=3>> <<5*3=15>>',
 '<<200*3=600>> <<600*0.4=240>>',
 '<<1/2*100=50>> <<3/5*50=30>> <<50+30=80>> <<100-80=20>>',
 '<<40*2=80>> <<80*2=160>>',
 '<<12*2=24>> <<4*1=4>> <<4*3=12>> <<24+4+12=40>> <<40/4=10>>',
 '<<2*2.25=4.50>> <<4.50+3.50+4+3.50=15.00>> <<2*2.50=5.00>> <<15+5+3.50=23.50>>',
 '<<28/4=7>> <<3*7+1=22>>']

In [246]:
labels_cot

['<<4-2=2>> <<2/.5=4>> <<12/4=3>> <<100*3=300>>',
 '<<1.5*2=3>> <<3+2.5=5.5>> <<1.5+3+5.5=10>>',
 '<<30/100*2000=600>> <<2000-600=1400>>',
 '<<21/7=3>> <<5*3=15>>',
 '<<200*3=600>> <<600*.4=240>>',
 '<<1/2*100=50>> <<3/5*50=30>> <<50-30=20>>',
 '<<40/2=20>> <<20/2=10>>',
 '<<12*2=24>> <<4*1=4>> <<3*4=12>> <<24+4+12=40>> <<12+4+4=20>> <<40/20=2>>',
 '<<2*2.25=4.50>> <<2*4=8.00>> <<2*2.50=5.00>> <<4.50+8.00+.50+5.00+3.50+3.50=25.00>>',
 '<<32=32>> <<8=8>>']

In [247]:
preds_ans

['300', '10', '1400', '15', '240', '20', '160', '10', '', '22']

In [248]:
labels_ans

['300', '10', '1400', '15', '240', '20', '10', '2', '25', '25']


### teacher forcing


In [193]:
labels_masks.shape, preds.shape

(torch.Size([10, 172]), (10, 173))

In [249]:
# preds = eval_pred.predictions.argmax(axis=-1)[:, :-1]
# labels = eval_pred.label_ids[:, 1:]

preds = out.logits.argmax(dim=-1).cpu().numpy()[:, :-1]
labels = collated['labels'][:, 1:]

labels_masks = labels > 0
# labels_masks[:, :1] = False
preds_full = [p[m][1:] for p, m in zip(preds, labels_masks)]
labels_full = [lab[m][1:] for lab, m in zip(labels, labels_masks)]

print(len(preds_full), len(labels_full))

preds_full_text = tokenizer.batch_decode(preds_full, add_special_tokens=True)
labels_full_text = tokenizer.batch_decode(labels_full, add_special_tokens=True)

preds_cot = [extract_cot(p) for p in preds_full_text]
preds_ans = [extract_answer(p) for p in preds_full_text]

labels_cot = [extract_cot(lab) for lab in labels_full_text]
labels_ans = [extract_answer(lab) for lab in labels_full_text]

acc_cot = np.mean([c == p for c, p in zip(preds_cot, labels_cot)])
acc_ans = np.mean([c == lab for c, lab in zip(preds_ans, labels_ans)])

print(f"Accuracy COT: {acc_cot}")
print(f"Accuracy Answer: {acc_ans}")

10 10
Accuracy COT: 0.3
Accuracy Answer: 0.9


In [251]:
preds_full_text

['<<4-2=2>> <<2/.5=4>> <<12/4=3>> <<100*3=300>><|endoftext|>300<|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><

In [252]:
labels_full_text

['<<4-2=2>> <<2/.5=4>> <<12/4=3>> <<100*3=300>><|endoftext|>300<|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><

In [253]:
preds_cot

['<<4-2=2>> <<2/.5=4>> <<12/4=3>> <<100*3=300>>',
 '<<1.5*2=3>> <<3+2.5=5.5>> <<1.5+3+5.5=10>>',
 '<<2000/100*2000=600>> <<2000-600=1400>>',
 '<<21/7=3>> <<5*3=15>>',
 '<<200*3=600>> <<600*4=240>>',
 '<<1/2*100=50>> <<3/5*50=30>> <<50+30=20>>',
 '<<40*2=20>> <<20/2=10>>',
 '<<12*2=24>> <<4*1=4>> <<4*4=12>> <<24+4+12=40>> <<40+4+12=20>> <<40/20=2>>',
 '<<2*2.25=4.50>> <<4*4=8.00>> <<2*2.50=5.00>> <<4.50+8+00+50+5+00+3.50=2.50=28.00>>',
 '<<28/32>> <<(.8>> <<8']

In [254]:
preds_cot

['<<4-2=2>> <<2/.5=4>> <<12/4=3>> <<100*3=300>>',
 '<<1.5*2=3>> <<3+2.5=5.5>> <<1.5+3+5.5=10>>',
 '<<2000/100*2000=600>> <<2000-600=1400>>',
 '<<21/7=3>> <<5*3=15>>',
 '<<200*3=600>> <<600*4=240>>',
 '<<1/2*100=50>> <<3/5*50=30>> <<50+30=20>>',
 '<<40*2=20>> <<20/2=10>>',
 '<<12*2=24>> <<4*1=4>> <<4*4=12>> <<24+4+12=40>> <<40+4+12=20>> <<40/20=2>>',
 '<<2*2.25=4.50>> <<4*4=8.00>> <<2*2.50=5.00>> <<4.50+8+00+50+5+00+3.50=2.50=28.00>>',
 '<<28/32>> <<(.8>> <<8']

In [255]:
preds_ans

['300', '10', '1400', '15', '240', '20', '10', '2', '25', '']

In [256]:
labels_ans

['300', '10', '1400', '15', '240', '20', '10', '2', '25', '25']

### Older pretokenize

In [ ]:
# from torch.nn.utils.rnn import pad_sequence
# id_pad_value = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else tokenizer.eos_token_id
# if args.use_cot in (False, None):
#     inputs_key = 'examples_nocot'
#     labels_key = 'labels_nocot'
# else:
#     inputs_key = 'examples_all'
#     labels_key = 'labels_all'
    
# def collate_fn(batch):
#     input_ids = [torch.tensor(b[inputs_key]) for b in batch]
#     labels = [torch.tensor(b[labels_key]) for b in batch]
#     attention_mask = [torch.ones_like(b, dtype=int) for b in input_ids]
#     # labels_mask defines which input_ids participate in loss calculation
#     labels_mask = [torch.sign(torch.tensor(b[labels_key])) for b in batch]


#     input_ids = pad_sequence(input_ids, padding_value=id_pad_value, batch_first=True)
#     labels = pad_sequence(labels, padding_value=id_pad_value, batch_first=True)
#     attention_mask = pad_sequence(attention_mask, padding_value=0, batch_first=True)
#     labels_mask = pad_sequence(labels_mask, padding_value=0, batch_first=True)

#     collated = {'input_ids': input_ids,
#                 'labels': labels, 
#                 'attention_mask': attention_mask,
#                 }
#     if args.num_mem_tokens is not None:
#         # add labels mask only for RMT, ARMT
#         collated['labels_mask'] = labels_mask.bool()
#     return collated


In [ ]:
# def extract_cot(text):
#     if '<|endoftext|>' not in text:
#         return ''
#     else:
#         return text.split('<|endoftext|>')[0].strip()

# def extract_answer(text):
#     if '####' not in text:
#         return ''
#     else:
#         ans = text.split('####')[-1]
#         ans = ans.split('<|endoftext|>')[0]
#         return ans.strip()
        
# def compute_accuracy(eval_pred):
#     preds = eval_pred.predictions[:, :-1]
#     labels = eval_pred.label_ids[:, 1:]
#     # inputs = eval_pred.inputs
#     # losses = eval_pred.losses

#     # labels = collated['labels'][:, 1:]

#     labels_masks = labels > 0
#     preds_full = [p[m] for p, m in zip(preds, labels_masks)]
#     labels_full = [l[m] for l, m in zip(labels, labels_masks)]

#     preds_full_text = tokenizer.batch_decode(preds_full, add_special_tokens=True)
#     labels_full_text = tokenizer.batch_decode(labels_full, add_special_tokens=True)

#     preds_cot = [extract_cot(p) for p in preds_full_text]
#     preds_ans = [extract_answer(p) for p in preds_full_text]

#     labels_cot = [extract_cot(l) for l in labels_full_text]
#     labels_ans = [extract_answer(l) for l in labels_full_text]
    
#     # Calculate accuracy only on the unignored tokens
#     acc_cot = np.mean([c == l for c, l in zip(preds_cot, labels_cot)])
#     acc_ans = np.mean([c == l for c, l in zip(preds_ans, labels_ans)])

#     return {'accuracy_cot': acc_cot, 'accuracy_ans': acc_ans}

In [39]:
# predictions = eval_pred.predictions
# label_ids = eval_pred.label_ids
# inputs = eval_pred.inputs
# losses = eval_pred.losses
# # elements = (self.predictions, self.label_ids)

In [40]:
batch = [valid_dataset[i] for i in range(10)]
collated = collate_fn(batch)

out = model(**collated)
print(out.keys())

odict_keys(['loss', 'logits', 'past_key_values'])


In [41]:
out.loss

tensor(0.0672, grad_fn=<NllLossBackward0>)

In [42]:
preds = out.logits.argmax(dim=-1).cpu().numpy()
preds.shape

(10, 175)

In [43]:
preds_text = tokenizer.batch_decode(preds, add_special_tokens=True)

In [44]:
# preds_text[0].split('<|endoftext|>')

In [46]:
# ''.split('<|endoftext|>')[1]

In [47]:
labels = collated['labels'][:, 1:]

labels_masks = labels > 0
preds_full = [p[m] for p, m in zip(preds[:, :-1], labels_masks)]
labels_full = [l[m] for l, m in zip(labels, labels_masks)]

preds_full_text = tokenizer.batch_decode(preds_full, add_special_tokens=True)
labels_full_text = tokenizer.batch_decode(labels_full, add_special_tokens=True)

preds_cot = [extract_cot(p) for p in preds_full_text]
preds_ans = [extract_answer(p) for p in preds_full_text]

labels_cot = [extract_cot(l) for l in labels_full_text]
labels_ans = [extract_answer(l) for l in labels_full_text]

In [48]:
# labels = collated['labels'][:, 1:]

# labels_masks = labels > 0
# preds_full = [p[m] for p, m in zip(preds[:, :-1], labels_masks)]
# labels_full = [l[m] for l, m in zip(labels, labels_masks)]

# preds_full_text = tokenizer.batch_decode(preds_full, add_special_tokens=True)
# labels_full_text = tokenizer.batch_decode(labels_full, add_special_tokens=True)

# preds_cot = [p.split('<|endoftext|>')[0].strip() for p in preds_full_text]
# labels_cot = [l.split('<|endoftext|>')[0].strip() for l in labels_full_text]

# # preds_ans = [p.split('<|endoftext|>')[1].strip()[4:] for p in preds_full_text]
# # labels_ans = [l.split('<|endoftext|>')[1].strip()[4:] for l in labels_full_text]

# preds_ans = [p.split('####')[1].strip()[4:] for p in preds_full_text]
# labels_ans = [l.split('####')[1].split('astrip()[4:] for l in labels_full_text]



In [49]:
tokenizer.batch_decode(collated['input_ids'])

['John cuts his grass to 2 inches.  It grows .5 inches per month.  When it gets to 4 inches he cuts it back down to 2 inches.  It cost $100 to get his grass cut.  How much does he pay per year?<|endoftext|><<4-2=2>> <<2/.5=4>> <<12/4=3>> <<100*3=300>><|endoftext|>300<|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|end

In [50]:
labels_full_text

['<|endoftext|><<4-2=2>> <<2/.5=4>> <<12/4=3>> <<100*3=300>><|endoftext|>300<|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><

In [51]:
preds_cot

['', '', '', '', '', '', '', '', '', '']

In [52]:
labels_cot

['', '', '', '', '', '', '', '', '', '']

In [53]:
labels_ans

['', '', '', '', '', '', '', '', '', '']

In [54]:
preds_ans

['', '', '', '', '', '', '', '', '', '']

In [27]:
labels_text
[0].split('<|endoftext|>')

NameError: name 'labels_text' is not defined

In [28]:
for p, l, m in zip(preds, collated['labels'], collated['labels_mask']):
    print(p[m], l[m])

KeyError: 'labels_mask'

In [42]:
preds[0]

array([   11,   860,   860,   604,  1635,   657,   657,   767,   657,
         642,   642,   642,   642,   718,   352,  1343,   657,   657,
         718,   604,   860,   657,   357,   642,   642,   352,   352,
         352,   352,  1267,  1343,   657,   657,   642,   860,   657,
         767,   657,   357,   642,   642,   718,   657,   362,   807,
         657,  1267,  1343,   657,   657,   657,   657,   718,   604,
         860,   657,   220, 50256,  1303, 21017,   642,   642,   718,
         657,   807,   362,   657,   352,   220, 50256,  1303])

In [27]:
pred_texts = tokenizer.batch_decode(preds, skip_special_tokens=False)
print(pred_texts)

[', 9 9 4 * 0 0 7 0 5 5 5 5 6 1 + 0 0 6 4 9 0 ( 5 5 1 1 1 1 ) + 0 0 5 9 0 7 0 ( 5 5 6 0 2 8 0 ) + 0 0 0 0 6 4 9 0 <|endoftext|> #### 5 5 6 0 8 2 0 1 <|endoftext|> #', ' the 0 1 6 0 1 0 8 3 6 6 7 1 1 3 + 0 4 8 7 0 2 ( 6 1 0 9 3 2 ) + 0 0 4 8 7 0 2 ( 6 1 4 7 1 3 2 ) + 0 0 0 2 7 3 6 3 <|endoftext|> #### 6 1 4 9 8 6 8 3 <|endoftext|> #', ' the 0 8 4 + 2 8 3 0 8 8 0 0 5 7 + 0 4 8 3 4 8 ( 8 4 8 8 1 9 ) + 0 0 6 7 3 9 0 ( 8 4 4 6 5 8 1 ) + 0 0 0 2 3 6 5 6 <|endoftext|> #### 8 4 4 8 8 4 7 6 <|endoftext|> #', ', 2 3 2\n 0 0 0 1 9 9 0 9 2 1 + 0 2 1 2 7 1 ( 9 2 0 5 8 1 ) + 0 0 8 1 8 5 2 ( 9 2 8 6 6 7 2 ) + 0 0 0 5 1 5 1 2 <|endoftext|> #### 9 2 8 1 8 2 4 2 <|endoftext|> #', ' the 1 - 0 0 0 1 3 2 0 0 4 6 8 5 + 0 0 5 6 6 3 ( 0 4 1 5 2 4 ) + 0 0 0 8 9 3 4 ( 0 4 1 3 2 8 4 ) + 0 0 0 0 5 6 6 3 <|endoftext|> #### 0 4 1 3 7 4 1 4 <|endoftext|> #', ', 4 5 1 0 8 2 8 1 2 2 5 2 4 2 + 0 9 8 1 8 1 ( 2 4 1 6 0 2 ) + 0 0 4 0 5 8 4 ( 2 4 5 6 5 0 5 ) + 0 0 0 1 4 4 2 4 <|endoftext|> #### 2 4 5 7 9 4 7 4 <|endoftext|

In [28]:
labels_texts = tokenizer.batch_decode(collated['input_ids'], skip_special_tokens=False)
print(labels_texts)

[' 5 6 3 2 * 7 4 3 4 <|endoftext|> 5 5 5 6 1 + 0 0 6 4 9 0 ( 5 5 1 1 1 1 ) + 0 0 5 9 0 7 0 ( 5 5 6 0 2 8 0 ) + 0 0 0 0 6 4 9 0 <|endoftext|> #### 5 5 6 0 8 2 0 1 <|endoftext|>', ' 6 9 1 5 * 6 4 4 7 <|endoftext|> 6 7 1 1 3 + 0 4 8 7 0 2 ( 6 1 0 9 3 2 ) + 0 0 4 8 7 0 2 ( 6 1 4 7 1 3 2 ) + 0 0 0 2 7 3 6 3 <|endoftext|> #### 6 1 4 9 8 6 8 3 <|endoftext|>', ' 6 7 3 9 * 8 9 1 7 <|endoftext|> 8 0 0 5 7 + 0 4 8 3 4 8 ( 8 4 8 8 1 9 ) + 0 0 6 7 3 9 0 ( 8 4 4 6 5 8 1 ) + 0 0 0 2 3 6 5 6 <|endoftext|> #### 8 4 4 8 8 4 7 6 <|endoftext|>', ' 3 0 3 4 * 3 4 6 5 <|endoftext|> 9 0 9 2 1 + 0 2 1 2 7 1 ( 9 2 0 5 8 1 ) + 0 0 8 1 8 5 2 ( 9 2 8 6 6 7 2 ) + 0 0 0 5 1 5 1 2 <|endoftext|> #### 9 2 8 1 8 2 4 2 <|endoftext|>', ' 0 3 3 7 * 8 5 6 5 <|endoftext|> 0 4 6 8 5 + 0 0 5 6 6 3 ( 0 4 1 5 2 4 ) + 0 0 0 8 9 3 4 ( 0 4 1 3 2 8 4 ) + 0 0 0 0 5 6 6 3 <|endoftext|> #### 0 4 1 3 7 4 1 4 <|endoftext|>', ' 3 6 0 6 * 4 3 8 7 <|endoftext|> 2 5 2 4 2 + 0 9 8 1 8 1 ( 2 4 1 6 0 2 ) + 0 0 4 0 5 8 4 ( 2 4 5 6 5 0 5 ) + 0 0 

In [29]:
collated['input_ids'].shape

torch.Size([10, 71])

In [30]:
pred_texts[0]

', 9 9 4 * 0 0 7 0 5 5 5 5 6 1 + 0 0 6 4 9 0 ( 5 5 1 1 1 1 ) + 0 0 5 9 0 7 0 ( 5 5 6 0 2 8 0 ) + 0 0 0 0 6 4 9 0 <|endoftext|> #### 5 5 6 0 8 2 0 1 <|endoftext|> #'

In [33]:
preds[0]

array([   11,   860,   860,   604,  1635,   657,   657,   767,   657,
         642,   642,   642,   642,   718,   352,  1343,   657,   657,
         718,   604,   860,   657,   357,   642,   642,   352,   352,
         352,   352,  1267,  1343,   657,   657,   642,   860,   657,
         767,   657,   357,   642,   642,   718,   657,   362,   807,
         657,  1267,  1343,   657,   657,   657,   657,   718,   604,
         860,   657,   220, 50256,  1303, 21017,   642,   642,   718,
         657,   807,   362,   657,   352,   220, 50256,  1303])

In [ ]:
collated['input_ids'][0]

torch.Size([71])

In [38]:
for p, t in zip(preds[0], collated['input_ids'][0][1:]):
    # print(p, tokenizer.decode([p]), t, tokenizer.decode([t]))
    print(tokenizer.decode([p]), tokenizer.decode([t]))

,  6
 9  3
 9  2
 4  *
 *  7
 0  4
 0  3
 7  4
 0  
 5 <|endoftext|>
 5  5
 5  5
 5  5
 6  6
 1  1
 +  +
 0  0
 0  0
 6  6
 4  4
 9  9
 0  0
 (  (
 5  5
 5  5
 1  1
 1  1
 1  1
 1  1
 )  )
 +  +
 0  0
 0  0
 5  5
 9  9
 0  0
 7  7
 0  0
 (  (
 5  5
 5  5
 6  6
 0  0
 2  2
 8  8
 0  0
 )  )
 +  +
 0  0
 0  0
 0  0
 0  0
 6  6
 4  4
 9  9
 0  0
   
<|endoftext|> <|endoftext|>
 #  #
### ###
 5  5
 5  5
 6  6
 0  0
 8  8
 2  2
 0  0
 1  1
   
<|endoftext|> <|endoftext|>
